In [ ]:
! pip install pyprind fancyimpute

In [ ]:
import argparse
import pyprind

import numpy as np
import pandas as pd

from scipy.spatial.distance import cdist
from scipy.interpolate import interp1d
from scipy import stats

from fancyimpute import KNN

In [ ]:
PROCESS_RAW = True # If specified, additionally save trajectories without normalized features
SAVE_INTERMEDIATE = True # "If specified, save off intermediate tables used to construct final patient table

In [ ]:
print('Loading processed files created from database using "preprocess.py"')
abx           = pd.read_csv('processed_files/abx.csv',           sep = '|')
culture       = pd.read_csv('processed_files/culture.csv',       sep = '|')
microbio      = pd.read_csv('processed_files/microbio.csv',      sep = '|')
demog         = pd.read_csv('processed_files/demog.csv',         sep = '|')
ce010         = pd.read_csv('processed_files/ce010000.csv',      sep = '|')
ce1020        = pd.read_csv('processed_files/ce1000020000.csv',  sep = '|')
ce2030        = pd.read_csv('processed_files/ce2000030000.csv',  sep = '|')
ce3040        = pd.read_csv('processed_files/ce3000040000.csv',  sep = '|')
ce4050        = pd.read_csv('processed_files/ce4000050000.csv',  sep = '|')
ce5060        = pd.read_csv('processed_files/ce5000060000.csv',  sep = '|')
ce6070        = pd.read_csv('processed_files/ce6000070000.csv',  sep = '|')
ce7080        = pd.read_csv('processed_files/ce7000080000.csv',  sep = '|')
ce8090        = pd.read_csv('processed_files/ce8000090000.csv',  sep = '|')
ce90100       = pd.read_csv('processed_files/ce90000100000.csv', sep = '|')
MV            = pd.read_csv('processed_files/mechvent.csv',      sep = '|')
inputpreadm   = pd.read_csv('processed_files/preadm_fluid.csv',  sep = '|')
#inputMV       = pd.read_csv('processed_files/fluid_mv.csv',      sep = '|')
#inputCV       = pd.read_csv('processed_files/fluid_cv.csv',      sep = '|')
inputMV       = pd.read_csv('processed_files/fluid_align_with_snuh_mv.csv',      sep = '|')
inputCV       = pd.read_csv('processed_files/fluid_align_with_snuh_cv.csv',      sep = '|')
#vasoMV        = pd.read_csv('processed_files/vaso_mv.csv',       sep = '|')
#vasoCV        = pd.read_csv('processed_files/vaso_cv.csv',       sep = '|')
vasoMV        = pd.read_csv('processed_files/vaso_align_with_snuh_mv.csv',       sep = '|')
vasoCV        = pd.read_csv('processed_files/vaso_align_with_snuh_cv.csv',       sep = '|')
UOpreadm      = pd.read_csv('processed_files/preadm_uo.csv',     sep = '|')
UO            = pd.read_csv('processed_files/uo.csv',            sep = '|')
note          = pd.read_csv('processed_files/noteevents.csv',    sep = '|')
labU          = [pd.read_csv('processed_files/labs_ce.csv', sep = '|') , pd.read_csv('processed_files/labs_le.csv', sep = '|')]

labU[1].rename(columns = {'timestp': 'charttime'}, inplace=True) 
labU = pd.concat(labU, sort=False, ignore_index=True)

# 1. Initial data manipulations & get icustay_id

In [ ]:
microbio['charttime'] = microbio['charttime'].fillna(microbio['chartdate'])
del microbio['chartdate']

note['charttime'] = note['charttime'].fillna(note['chartdate'])
del note['chartdate']

bacterio = pd.concat([microbio, culture], sort=False, ignore_index=True)

demog['morta_90'].fillna(0, inplace=True)
demog['morta_hosp'].fillna(0, inplace=True)
demog['elixhauser'].fillna(0, inplace=True)

# Keep only the first icustay of an admission (CRITICAL FIX FROM MATLAB CODE)
demog = demog.drop_duplicates(subset=['admittime','dischtime'],keep='first')

# Get list of all icustayids since that's what we iterate over through the rest of this script
# The old code had a continuous range of icustayids so it was easy to loop through them with a range(numIDS),
# Since we're only keeping the first icustay of a patient's admission, this is now different...
icustayidlist = list(demog.icustay_id.values)

# 2. Calculate the accurate readmission using the demographics data 

In [ ]:
# Calculate the accurate readmission using the demographics data 
# (the SQL code from Komorowski, et al incorrectly cumulatively counts how many icu stays each patient has (preprocess.py:line 414) 
# and does a coarse boolean check if this number is >1). A readmission is now correctly defined by 
# whether the patient has returned to the ICU within 30 days of being previously discharged.

# This is done by grouping all the discharge times for each patient and using them in a comparison 
# with the current row's admission time to see if it's within the 30 day cutoff
subj_dischtime_list = demog.sort_values(by='admittime').groupby('subject_id').apply(lambda df: np.unique(df.dischtime.values)) # Create list of discharge times for each patient (output is a dict keyed by 'subject_id')

def determine_readmission(s, dischtimes=subj_dischtime_list,cutoff=3600*24*30):
    '''
    determine_readmisson evaluates each row of the provided dataframe (designed to operate on the demographics table)
    and chooses whether the current admission occurs within the cutoff of the previous discharge 
    (here, cutoff=30 days is the default)
    '''
    subject, admission, discharge = s[['subject_id','admittime','dischtime']]
    
    # Check for readmission
    subj_stay_idx = np.where(dischtimes[subject]==discharge)[0][0]
    s['re_admission'] = 0
    if subj_stay_idx > 0:
        if (admission - dischtimes[subject][subj_stay_idx-1]) <= cutoff:
            s['re_admission'] = 1
            
    return s
# Apply the above function to determine the appropriate readmissions
demog = demog.apply(determine_readmission,axis=1)

# 3. helper function for imputation anomal processing

In [ ]:
def SAH(input, vitalslab_hold, adjust=0):
    '''Matthieu Komorowski - Imperial College London 2017 
    will copy a value in the rows below if the missing values are within the
    hold period for this variable (e.g. 48h for weight, 2h for HR...)
    vitalslab_hold = 2x55 cell (with row1 = strings of names ; row 2 = hold time)'''
    temp = np.copy(input)
    hold = vitalslab_hold.values[0, :]
    nrow, ncol = temp.shape

    lastcharttime = np.zeros(ncol)
    lastvalue = np.zeros(ncol)
    oldstayid = temp[0, 1]

    bar_SAH = pyprind.ProgBar(ncol-(3+adjust))
    for i in range(3+adjust,ncol):
        bar_SAH.update()
        for j in range(nrow):
            if oldstayid != temp[j, 1]:
                lastcharttime = np.zeros(ncol)
                lastvalue = np.zeros(ncol)
                oldstayid = temp[j, 1]
            if not np.isnan(temp[j, i]):
                lastcharttime[i] = temp[j, 2]
                lastvalue[i] = temp[j, i]
            if j > 0:
                if (np.isnan(temp[j, i])) and (temp[j, 1] == oldstayid) and ((temp[j, 2] - lastcharttime[i]) <= hold[i-(3+adjust)]*3600):
                    temp[j,i] = lastvalue[i]
    return temp

def fixgaps(x):
    '''FIXGAPS Linearly interpolates gaps in a time series
    YOUT=FIXGAPS(YIN) linearly interpolates over NaN
    in the input time series (may be complex), but ignores
    trailing and leading NaN.
    R. Pawlowicz 6/Nov/99'''
    y = np.copy(x)
    bd = np.isnan(x)
    gd = np.arange(len(x))[~bd]
    bd[:min(gd)] = False
    bd[max(gd)+1:] = False
    y[bd] = interp1d(gd,x[gd])(np.arange(len(x))[bd])
    return y

def deloutabove(a, col_no, a_max):
    a[a[:,col_no] > a_max, col_no] = np.nan 
    return a

def deloutbelow(a, col_no, a_min):
    a[a[:,col_no] < a_min, col_no] = np.nan 
    return a

# 4. Compute normalized rate of infusion

In [ ]:
# Compute normalized rate of infusion
# if 100 ml of hypertonic fluid (600 mosm/l) is given at 100 ml/h (given in 1h) it is 200 ml of NS equivalent
# so the normalized rate of infusion is 200 ml/h (different volume in same duration)
inputMV['norm_rate_of_infusion'] = inputMV['tev']*inputMV['rate']/inputMV['amount']

# 5. Fill-in missing ICUSTAY IDs in bacterio and ABX

In [ ]:
print('Filling-in missing ICUSTAY IDs in bacterio')
bar = pyprind.ProgBar(len(bacterio.index.tolist()))
# Raw Translation
for i in bacterio.index.tolist():
    bar.update()
    if np.isnan(bacterio.loc[i, 'icustay_id']):
        o         = bacterio.loc[i, 'charttime'] 
        subjectid = bacterio.loc[i, 'subject_id']
        hadmid    = bacterio.loc[i, 'hadm_id']
        ii        = demog.index[demog['subject_id'] == subjectid].tolist()
        jj        = demog.index[(demog['subject_id'] == subjectid) & (demog['hadm_id'] == hadmid)].tolist()
        for j in range(len(ii)):
            if (o >= demog.loc[ii[j], 'intime'] - 48*3600) and (o <= demog.loc[ii[j], 'outtime'] + 48*3600):
                bacterio.loc[i,'icustay_id'] = demog.loc[ii[j], 'icustay_id']
            elif len(ii)==1:   # If we cant confirm from admission and discharge time but there is only 1 admission: it's the one!!
                bacterio.loc[i,'icustay_id'] = demog.loc[ii[j], 'icustay_id']

print('Filling-in missing ICUSTAY IDs in bacterio - 2')                
bar = pyprind.ProgBar(len(bacterio.index.tolist()))
for i in bacterio.index.tolist():
    bar.update()
    if np.isnan(bacterio.loc[i, 'icustay_id']):
        subjectid = bacterio.loc[i, 'subject_id']
        hadmid    = bacterio.loc[i, 'hadm_id']
        jj        = demog.index[(demog['subject_id'] == subjectid) & (demog['hadm_id'] == hadmid)].tolist()
        if len(jj) == 1:
            bacterio.loc[i,'icustay_id'] = demog.loc[jj[0], 'icustay_id']

# Fill-in missing ICUSTAY IDs in Antibiotics administration
print('Filling-in missing ICUSTAY IDs in ABx')
bar = pyprind.ProgBar(len(abx.index.tolist()))
for i in abx.index.tolist():
    bar.update()
    if np.isnan(abx.loc[i,'icustay_id']):
        o      = abx.loc[i,'startdate']  #time of event
        hadmid = abx.loc[i,'hadm_id']
        ii     = demog.index[demog['hadm_id'] == hadmid].tolist()
        for j in range(len(ii)):
            if o >= demog.loc[ii[j],'intime'] - 48*3600 and o <= demog.loc[ii[j], 'outtime'] + 48*3600:
                abx.loc[i, 'icustay_id'] = demog.loc[ii[j], 'icustay_id']
            elif len(ii) == 1:   #if we cant confirm from admission and discharge time but there is only 1 admission: it's the one!!
                abx.loc[i, 'icustay_id'] = demog.loc[ii[j], 'icustay_id']
                

# 6. fill-in missing ICUSTAY IDs in Note

In [ ]:
note['text'] = '['+note['category'].astype(str) + ']' + note['text']
note.drop(['category'], axis=1, inplace=True)

In [ ]:
note['icustay_id'] = np.nan

In [ ]:
print('Filling-in missing ICUSTAY IDs in note')

bar = pyprind.ProgBar(len(note.index.tolist()))
# Raw Translation
for i in note.index.tolist():
    bar.update()
    if np.isnan(note.loc[i, 'icustay_id']):
        o         = note.loc[i, 'charttime'] 
        subjectid = note.loc[i, 'subject_id']
        hadmid    = note.loc[i, 'hadm_id']
        ii        = demog.index[demog['subject_id'] == subjectid].tolist()
        jj        = demog.index[(demog['subject_id'] == subjectid) & (demog['hadm_id'] == hadmid)].tolist()
        for j in range(len(ii)):
            if (o >= demog.loc[ii[j], 'intime'] - 48*3600) and (o <= demog.loc[ii[j], 'outtime'] + 48*3600):
                note.loc[i,'icustay_id'] = demog.loc[ii[j], 'icustay_id']
            elif len(ii)==1:   # If we cant confirm from admission and discharge time but there is only 1 admission: it's the one!!
                note.loc[i,'icustay_id'] = demog.loc[ii[j], 'icustay_id']

In [ ]:
print('Filling-in missing ICUSTAY IDs in note - 2')                
bar = pyprind.ProgBar(len(note.index.tolist()))
for i in note.index.tolist():
    bar.update()
    if np.isnan(note.loc[i, 'icustay_id']):
        subjectid = note.loc[i, 'subject_id']
        hadmid    = note.loc[i, 'hadm_id']
        jj        = demog.index[(demog['subject_id'] == subjectid) & (demog['hadm_id'] == hadmid)].tolist()
        if len(jj) == 1:
            note.loc[i,'icustay_id'] = demog.loc[jj[0], 'icustay_id']

In [ ]:
note

# 6. Find presumed onset of infection according to sepsis3 guidelines

In [ ]:
# METHOD:
# Loop through all administered antibiotics as soon as
# a sample is present within the time window break the loop.

print('Full ICU -- Finding presumed onset of infection according to sepsis3 guidelines')

onset = dict()
num_onset = 0
bar = pyprind.ProgBar(len(icustayidlist))
for icustayid in icustayidlist:
    bar.update()
    onset[icustayid] = np.zeros(3)
    ab = abx.loc[abx['icustay_id'] == icustayid, 'startdate']   # Start time of abx for this icustayid
    bact = bacterio.loc[bacterio['icustay_id'] == icustayid, 'charttime']   # Time of sample
    subj_bact = bacterio.loc[bacterio['icustay_id'] == icustayid,'subject_id'] 
    
    if len(ab) > 0 and len(bact) > 0:   # If we have data for both: proceed
        # Pairwise distances between antibiotic adminstration and requested cultures, in hours
        D = cdist(ab.values.reshape(ab.values.shape[0],1),bact.values.reshape(bact.values.shape[0],1))/3600  
        for i in range(D.shape[0]):  #looping through all rows of adminsitered antibiotics, from early to late
            M, I = np.min(D[i,:]), np.argmin(D[i,:])        # minimum distance in this row
            ab1 = ab.iloc[i]       # timestamp of this value in list of antibiotics
            bact1 = bact.iloc[I]   # timestamp in list of cultures
            if M <= 24 and ab1 <= bact1:      # if ab was first and delay < 24h
                onset[icustayid][0] = subj_bact.iloc[0] #subject_id
                onset[icustayid][1] = icustayid # icustay_id
                onset[icustayid][2] = ab1     # Onset of infection = abx time
                num_onset += 1
                break
            elif M <= 72 and ab1 >= bact1:    # elseif sample was first and delay < 72h
                onset[icustayid][0] = subj_bact.iloc[0]   
                onset[icustayid][1] = icustayid
                onset[icustayid][2] = bact1       # Onset of infection = sample time
                num_onset += 1
                break

# Sum of records found
print('Full ICU -- Number of preliminary, presumed septic trajectories: ', num_onset)

# 7. Replacing item_ids with column numbers from reference tables lab이랑 chartevent만

In [ ]:
print('Full ICU -- Replacing item_ids with column numbers from reference tables')

# Replace itemid in labs with column number
# This will accelerate process later
Reflabs = pd.read_csv("ReferenceFiles/Reflabs.tsv", sep = '\t', header=None)
Reflabs_values = np.unique(Reflabs.fillna(-10000))[1:]
Reflabs_id_dict = {}
for r in Reflabs_values:
    try:
        Reflabs_id_dict[r] = np.max(np.where(Reflabs.values == r)[0]) + 1 # for row: +1 due to Index correction python        
    except:
        print(r)
        break
itemid_col = labU.columns.tolist().index('itemid')
labU_temp = labU.values
for c in range(labU_temp.shape[0]):
    labU_temp[c,itemid_col] = Reflabs_id_dict[labU_temp[c,itemid_col]]
for i, c in enumerate(labU.columns.tolist()):
    labU.loc[:,c] = labU_temp[:,i]


# Replace itemid in vitals with col number
Refvitals = pd.read_csv("ReferenceFiles/Refvitals.tsv", sep = '\t', header=None)
Refvitals_values = np.unique(Refvitals.fillna(-10000))[1:]
Refvitals_id_dict = {}
for r in Refvitals_values:
    Refvitals_id_dict[r] = np.max(np.where(Refvitals.values == r)[0]) + 1 # +1 due to index correction for Python from MATLAB
ce_dfs = [ce010, ce1020, ce2030, ce3040, ce4050, ce5060, ce6070, ce7080, ce8090, ce90100]
for ce_df in ce_dfs:
    itemid_col = ce_df.columns.tolist().index('itemid')
    ce_df_temp = ce_df.values
    for c in range(ce_df_temp.shape[0]):
        ce_df_temp[c,itemid_col] = Refvitals_id_dict[ce_df_temp[c,itemid_col]]
    for i, c in enumerate(ce_df.columns.tolist()):
        ce_df.loc[:,c] = ce_df_temp[:,i]

# 8. Initial reformat with chartevents, labs and mechvent

In [ ]:
print(' Full ICU --  Making an array with all unique charttime (1 per row) and all items in columns.')
reformat = np.nan*np.ones((2000000,68))  # Final table  
qstime = dict()
winb4 = 25   # Lower limit for inclusion of data (24h before time flag)
winaft = 49  # Upper limit (48h after)
irow = 0  # Recording row for summary table
bar = pyprind.ProgBar(len(icustayidlist))
for icustayid in icustayidlist:
    qstime[icustayid] = np.zeros(4)
    bar.update()
    qst = onset[icustayid][2] #flag for presumed infection
    if qst > 0:  # if we have a flag
        d1 = demog.loc[demog['icustay_id'] == icustayid, ['age', 'dischtime']].values[0] # Age of patient + discharge time
        if d1[0] > 6574:  # If older than 18 years old
            # CHARTEVENTS
            if (icustayid-200000) < 10000:
                temp=ce010
            elif (icustayid-200000) < 20000:
                temp=ce1020
            elif (icustayid-200000) < 30000:
                temp=ce2030
            elif (icustayid-200000) < 40000:
                temp=ce3040
            elif (icustayid-200000) < 50000:
                temp=ce4050
            elif (icustayid-200000) < 60000:
                temp=ce5060
            elif (icustayid-200000) < 70000:
                temp=ce6070
            elif (icustayid-200000) < 80000:
                temp=ce7080
            elif (icustayid-200000) < 90000:
                temp=ce8090
            else:
                temp=ce90100
            temp = temp[temp['icustay_id'] == icustayid]

            ii = (temp['charttime'] >= qst - (winb4+4)*3600) & (temp['charttime'] <= qst + (winaft+4)*3600) # Time period of interest -4h and +4h
            temp = temp.loc[ii]   # Only time period of interest

            # LABEVENTS
            ii = labU['icustay_id'] == icustayid
            temp2 = labU.loc[ii]
            ii = (temp2['charttime'] >= qst - (winb4+4)*3600) & (temp2['charttime'] <= qst + (winaft+4)*3600) # Time period of interest -4h and +4h
            temp2 = temp2.loc[ii]   # Only time period of interest

            # Mech Vent + ?extubated
            ii = MV['icustay_id'] == icustayid
            temp3 = MV.loc[ii]
            ii = (temp3['charttime'] >= qst - (winb4+4)*3600) & (temp3['charttime'] <= qst + (winaft+4)*3600) # Time period of interest -4h and +4h
            temp3 = temp3.loc[ii]   #only time period of interest
            t = np.unique(pd.concat([temp['charttime'], temp2['charttime'], temp3['charttime']], ignore_index=True).values) # List of unique timestamps from all 3 sources / sorted in ascending order
 
            if len(t) > 0:
                for i in range(len(t)):
                    #CHARTEVENTS
                    ii = temp['charttime'] == t[i]
                    col = temp.loc[ii,'itemid']
                    value = temp.loc[ii,'valuenum']
                    reformat[irow, 0] = i+1 #timestep  
                    reformat[irow, 1] = icustayid
                    reformat[irow, 2] = t[i] #charttime
                    reformat[irow, 2+col.astype(int).values] = value.values # Store available values

                    # LAB VALUES
                    ii = temp2['charttime'] == t[i]
                    col = temp2.loc[ii, 'itemid']
                    value = temp2.loc[ii, 'valuenum']
                    reformat[irow, 30+col.astype(int).values] = value.values  # Store available values
                
                    # Mechanical Ventilation  
                    ii = temp3['charttime'] == t[i]
                    if np.nansum(ii) > 0:
                        col = temp3.loc[ii, 'mechvent']
                        value = temp3.loc[ii, 'extubated']
                        reformat[irow, 66] = col.values[0] # Store available values
                        reformat[irow, 67] = value.values[0] # Store available values
                    else:
                        reformat[irow, 66]= np.nan
                        reformat[irow, 67]= np.nan
                    irow += 1

                qstime[icustayid][0] = qst # Flag for presumed infection / this is time of sepsis if SOFA >=2 for this patient
                # SAVE FIRST and LAST TIMESTAMPS, in QSTIME, for each ICUSTAYID
                qstime[icustayid][1] = t[0]   # First timestamp
                qstime[icustayid][2] = t[-1]  # Last timestamp
                qstime[icustayid][3] = d1[1]  # Discharge time

reformat = np.delete(reformat, range(irow, len(reformat)) ,axis=0)  # Delete unused rows

# 9. outliers

In [ ]:
print('Full ICU -- Handling outliers')

# Weight
reformat = deloutabove(reformat, 4, 300) 

# Heart Rate
reformat = deloutabove(reformat, 7, 250)

# Blood Pressure
reformat = deloutabove(reformat, 8, 300)
reformat = deloutbelow(reformat, 9, 0) 
reformat = deloutabove(reformat, 9, 200)
reformat = deloutbelow(reformat, 10, 0) 
reformat = deloutabove(reformat, 10, 200) 

# Respiratory Rate
reformat = deloutabove(reformat, 11, 80) 

# SpO2
reformat = deloutabove(reformat, 12, 150) 
reformat[reformat[:, 12]>100, 12] = 100

# Temperature
reformat[(reformat[:, 13] > 90) & (np.isnan(reformat[:, 14])), 14] = reformat[(reformat[:, 13] > 90) & (np.isnan(reformat[:, 14])), 13]
reformat = deloutabove(reformat, 13, 90) 

# Interface / is in col 22
# FiO2
reformat = deloutabove(reformat, 22, 100) 
reformat[reformat[:, 22] < 1 , 22] = reformat[reformat[:,22] < 1 , 22]*100

reformat = deloutbelow(reformat, 22, 20) 
reformat = deloutabove(reformat, 23, 1.5)

# O2 FLOW
reformat = deloutabove(reformat, 24, 70)

#PEEP
reformat=deloutbelow(reformat, 25, 0) 
reformat=deloutabove(reformat, 25, 40) 

# Total Volume
reformat=deloutabove(reformat, 26, 1800)

# Mean Volume
reformat=deloutabove(reformat, 27, 50)

# Potassium
reformat=deloutbelow(reformat, 31, 1) 
reformat=deloutabove(reformat, 31, 15) 

# Sodium
reformat=deloutbelow(reformat, 32, 95) 
reformat=deloutabove(reformat, 32, 178)

# Chloride
reformat=deloutbelow(reformat, 33, 70) 
reformat=deloutabove(reformat, 33, 150)

# Glucose
reformat=deloutbelow(reformat, 34, 1) 
reformat=deloutabove(reformat, 34, 1000)

# Creatinine
reformat=deloutabove(reformat, 36, 150)

# Magnesium
reformat=deloutabove(reformat, 37, 10)

# Calcium
reformat=deloutabove(reformat, 38, 20)

# Ionized Calcium
reformat=deloutabove(reformat, 39, 5) 

# CO2
reformat=deloutabove(reformat, 40, 120) 

# SGPT/SGOT
reformat=deloutabove(reformat, 41, 10000) 
reformat=deloutabove(reformat, 42, 10000) 

# Hb/Ht
reformat=deloutabove(reformat, 49, 20) 
reformat=deloutabove(reformat, 50, 65) 

# White Blood Cells
reformat=deloutabove(reformat, 52, 500) 

# Platelets
reformat=deloutabove(reformat, 53, 2000)

# INR
reformat=deloutabove(reformat, 57, 20) 

# pH
reformat=deloutbelow(reformat, 58, 6.7) 
reformat=deloutabove(reformat, 58, 8) 

# pO2
reformat=deloutabove(reformat, 59, 700)

# pCO2
reformat=deloutabove(reformat, 60, 200)

# Base Excess
reformat=deloutbelow(reformat, 61, -50)

# Lactate
reformat=deloutabove(reformat, 62, 30) 

# 10. More data manipulation / imputation from existing values

In [ ]:
# Estimate GCS from RASS - data from Wesley JAMA 2003
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] >= 0), 5] = 15
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -1), 5] = 14
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -2), 5] = 12
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -3), 5] = 11
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -4), 5] = 6
reformat[(np.isnan(reformat[:, 5])) & (reformat[:, 6] == -5), 5] = 3

# FiO2
reformat[(~np.isnan(reformat[:, 22])) & (np.isnan(reformat[:, 23])), 23] = reformat[(~np.isnan(reformat[:, 22])) & (np.isnan(reformat[:, 23])), 22] / 100
reformat[(~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 22])), 22] = reformat[(~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 22])), 23] * 100

In [ ]:
print('Full ICU -- Doing sample and hold')
sample_and_hold = pd.read_csv('ReferenceFiles/sample_and_hold.csv', index_col = None)

reformatsah = SAH(reformat,sample_and_hold)  # Do SAH first to handle this task

In [ ]:
# NO FiO2, YES O2 flow, no interface OR cannula
ii = np.where((np.isnan(reformatsah[:, 22])) & (~np.isnan(reformatsah[:, 24])) & ((reformatsah[:, 21] == 0) | (reformatsah[:, 21] == 2)))[0] #As np.where returns a tuple
reformat[ii[reformatsah[ii, 24] <= 15], 22] = 70
reformat[ii[reformatsah[ii, 24] <= 12], 22] = 62
reformat[ii[reformatsah[ii, 24] <= 10], 22] = 55
reformat[ii[reformatsah[ii, 24] <= 8], 22]  = 50
reformat[ii[reformatsah[ii, 24] <= 6], 22]  = 44
reformat[ii[reformatsah[ii, 24] <= 5], 22]  = 40
reformat[ii[reformatsah[ii, 24] <= 4], 22]  = 36
reformat[ii[reformatsah[ii, 24] <= 3], 22]  = 32
reformat[ii[reformatsah[ii, 24] <= 2], 22]  = 28
reformat[ii[reformatsah[ii, 24] <= 1], 22]  = 24

# NO FiO2, NO O2 flow, no interface OR cannula
ii = np.where((np.isnan(reformatsah[:, 22])) & np.isnan(reformatsah[:, 24]) & ((reformatsah[:, 21] == 0) | (reformatsah[:, 21] == 2)))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 22] = 21

# NO FiO2, YES O2 flow, face mask OR.... OR ventilator (assume it's face mask)
ii = np.where((np.isnan(reformatsah[:, 22])) & (~np.isnan(reformatsah[:, 24])) & 
((reformatsah[:, 21] == 1) | (reformatsah[:, 21]==3) | (reformatsah[:, 21] == 4) | (reformatsah[:, 21] == 5) | (reformatsah[:, 21]==6) | (reformatsah[:, 21]==9) | (reformatsah[:, 21]==10)))[0]
reformat[ii[reformatsah[ii, 24]<=15], 22] = 75
reformat[ii[reformatsah[ii, 24]<=12], 22] = 69
reformat[ii[reformatsah[ii, 24]<=10], 22] = 66
reformat[ii[reformatsah[ii, 24]<=8], 22]  = 58
reformat[ii[reformatsah[ii, 24]<=6], 22]  = 40
reformat[ii[reformatsah[ii, 24]<=4], 22]  = 36

# NO FiO2, NO O2 flow, face mask OR ....OR ventilator
ii = np.where(np.isnan(reformatsah[:, 22]) & np.isnan(reformatsah[:, 24]) & ((reformatsah[:, 21] == 1) | (reformatsah[:, 21] == 3) | 
(reformatsah[:, 21] == 4) | (reformatsah[:, 21] == 5) | (reformatsah[:, 21] == 6) | (reformatsah[:, 21] == 9) | (reformatsah[:, 21] == 10)))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 22] = np.nan

# NO FiO2, YES O2 flow, Non rebreather mask
ii = np.where(np.isnan(reformatsah[:, 22]) & (~np.isnan(reformatsah[:, 24])) & (reformatsah[:, 21] == 7))[0]
reformat[ii[reformatsah[ii, 24] >= 10], 22] = 90
reformat[ii[reformatsah[ii, 24] >= 15], 22] = 100
reformat[ii[reformatsah[ii, 24] < 10], 22]  = 80
reformat[ii[reformatsah[ii, 24] <= 8], 22]  = 70
reformat[ii[reformatsah[ii, 24] <= 6], 22]  = 60

# NO FiO2, NO O2 flow, NRM
ii= np.where(np.isnan(reformatsah[:, 22]) & np.isnan(reformatsah[:, 24]) & (reformatsah[:, 21]==7))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 22] = np.nan

# Update FiO2 columns again
ii = (~np.isnan(reformat[:, 22])) & (np.isnan(reformat[:,23]))
reformat[ii, 23] = reformat[ii, 22]/100
ii = (~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 22]))
reformat[ii, 22] = reformat[ii, 23]*100

In [ ]:
# Blood Pressure
ii = (~np.isnan(reformat[:, 8])) & (~np.isnan(reformat[:, 9])) & np.isnan(reformat[:, 10])
reformat[ii, 10] = (3*reformat[ii, 9] - reformat[ii, 8])/2
ii = (~np.isnan(reformat[:, 8])) & (~np.isnan(reformat[:, 10])) & np.isnan(reformat[:, 9])
reformat[ii, 9] = (reformat[ii, 8] + 2*reformat[ii, 10])/3
ii = (~np.isnan(reformat[:, 9])) & (~np.isnan(reformat[:, 10])) & np.isnan(reformat[:, 8])
reformat[ii, 8] = 3*reformat[ii, 9] - 2*reformat[ii, 10]

# Temperature
# Some values recorded in the wrong column
ii = (reformat[:, 14] > 25) & (reformat[:, 14] < 45) # tempF close to 37deg??!
reformat[ii, 13] = reformat[ii, 14]
reformat[ii, 14] = np.nan
ii = reformat[:, 13] >70  # tempC > 70, likely recorded in Farenheit
reformat[ii, 14] = reformat[ii, 13]
reformat[ii, 13] = np.nan
ii = (~np.isnan(reformat[:, 13])) & np.isnan(reformat[:, 14])
reformat[ii, 14] = reformat[ii, 13]*1.8+32
ii = (~np.isnan(reformat[:, 14])) & np.isnan(reformat[:, 13])
reformat[ii, 13] = (reformat[ii, 14] - 32)/1.8

# Hb/Ht
ii = (~np.isnan(reformat[:,49])) & np.isnan(reformat[:, 50])
reformat[ii, 50] = (reformat[ii, 49] * 2.862) + 1.216
ii = (~np.isnan(reformat[:, 50])) & np.isnan(reformat[:, 49])
reformat[ii, 49] = (reformat[ii, 50] - 1.216)/2.862

# Bilirubin
ii = (~np.isnan(reformat[:, 43])) & np.isnan(reformat[:, 44])
reformat[ii, 44] = (reformat[ii, 43]*0.6934)-0.1752
ii = (~np.isnan(reformat[:, 44])) & np.isnan(reformat[:, 43])
reformat[ii, 43] = (reformat[ii, 44] + 0.1752)/0.6934

# 11. SAMPLE AND HOLD on RAW DATA

In [ ]:
print('Full ICU -- SAMPLE AND HOLD on RAW DATA')
reformat = SAH(reformat[:,0:68],sample_and_hold)

In [ ]:
# reformat_copy = reformat

In [ ]:
# np.allclose(reformat, reformat_copy, equal_nan=True)

# 12. Data combination

In [ ]:
#reformat = reformat_copy

In [ ]:
print('Full ICU -- Data combination')
# WARNING: the time window of interest has been defined above (here -24 -> +48)! 
timestep = 4  # Resolution of timesteps, in hours
irow = 0   
icustayidlist = np.unique(reformat[:,1]).astype(np.int32)
reformat2 = np.nan*np.ones((reformat.shape[0], 86))  # Output array: add one column for notes.
reformat2 = reformat2.astype(object)

num_patients = len(icustayidlist)  # Number of patients
# Adding 2 empty cols for future shock index=HR/SBP and P/F
reformat = np.hstack([reformat, np.nan*np.ones((reformat.shape[0], 2))])

In [ ]:
bar = pyprind.ProgBar(num_patients)
for i in range(len(icustayidlist)):
    bar.update()
    icustayid = icustayidlist[i] 
    
    #CHARTEVENTS AND LAB VALUES
    temp = reformat[reformat[:,1] == icustayid,:]   # subtable of interest
    beg = temp[0,2]   # timestamp of first record
    
    #IV FLUID STUFF
    iv = inputMV['icustay_id'] == icustayid         # rows of interest in inputMV
    input = inputMV[iv]                             # subset of interest
    iv = inputCV['icustay_id'] == icustayid         # rows of interest in inputCV
    input2 = inputCV[iv]                            # subset of interest
    startt = input['starttime']                     # start of all infusions and boluses
    endt = input['endtime']                         # end of all infusions and boluses
    rate = input['norm_rate_of_infusion']           # rate of infusion (is NaN for boluses) || corrected for tonicity
    pread = inputpreadm[inputpreadm['icustay_id'] == icustayid]['inputpreadm'] # preadmission volume

    input = input.values
    input2 = input2.values

    if len(pread) >0:  # store the value, if available
        totvol = np.nansum(pread)
    else:
        totvol=0   # if not documented: it's zero
       
    # Compute volume of fluid given before start of record!!!
    t0 = 0
    t1 = beg
    # Input from MetaVision (4 ways to compute)
    infu = np.nansum(rate*(endt-startt)*((endt<=t1) & (startt >= t0))/3600 + rate*(endt-t0)*((startt <= t0) & (endt <= t1) & (endt >= t0))/3600 +
                            rate*(t1-startt)*((startt >= t0) & (endt >=t1) & (startt <= t1))/3600 + rate*(t1-t0)*((endt >= t1) & (startt <= t0))/3600)
    # All boluses received during this timestep, from inputMV (need to check rate is NaN) and inputCV (simpler):
    bolus = np.nansum(input[np.isnan(input[:, 5]) & (input[:, 1] >= t0) & (input[:, 1] <= t1), 6]) + np.nansum(input2[(input2[:, 1] >= t0) & (input2[:, 1] <= t1), 4])
    totvol = np.nansum([totvol,infu,bolus]) 
    
    #########################################################################################
    #VASOPRESSORS    
    iv = vasoMV['icustay_id'] == icustayid  # rows of interest in vasoMV
    vaso1 = vasoMV[iv]    # subset of interest
    iv = vasoCV['icustay_id'] == icustayid   # rows of interest in vasoCV
    vaso2 = vasoCV[iv]    # subset of interest
    startv = vaso1['starttime'].values  # start of VP infusion
    endv = vaso1['endtime'].values      # end of VP infusions
    ratev = vaso1['rate_std'].values  # rate of VP infusion
    
    #########################################################################################
    #note
    noti = note['icustay_id'] == icustayid  # rows of interest in vasoMV
    note1 = note[noti]    # subset of interest

    # DEMOGRAPHICS / gender, age, elixhauser, re-admit, died in hosp?, died within
    # 48h of out_time (likely in ICU or soon after), died within 90d after admission?        
    demogi = demog['icustay_id'] == icustayid
    dem = np.array(
            list(demog.gender[demogi].values) +
            list(demog.age[demogi].values) +
            list(demog.elixhauser[demogi].values) +
            list(demog.re_admission[demogi].values) + 
            list(demog.morta_hosp[demogi].values) + 
            list(abs(demog.dod[demogi].values - demog.outtime[demogi].values) < (24*3600*2)) +
            list(demog.morta_90[demogi].values) + 
            [(qstime[icustayid][3] - qstime[icustayid][2])/3600]
            )
        
        
    #URINE OUTPUT
    iu = UO['icustay_id'] == icustayid   #rows of interest in inputMV
    output = UO[iu]    #subset of interest
    pread = UOpreadm[UOpreadm['icustay_id'] == icustayid-200000]['value'].values #preadmission UO ????????????????? Why no + 200000 for icustayid here?
    if len(pread) > 0:     #store the value, if available
        UOtot = np.nansum(pread)
    else:
        UOtot = 0
    #adding the volume of urine produced before start of recording!    
    UOnow = np.nansum(output[(output['charttime']>=t0) & (output['charttime'] <= t1)]['value'].values) #t0 and t1 defined above
    UOtot = np.nansum([UOtot, UOnow])
    
    
    for j in range(0, 79, timestep): #0:timestep:79 #-28 until +52 = 80 hours in total
        t0 = 3600*j + beg   #left limit of time window
        t1 = 3600*(j + timestep) + beg   #right limit of time window
        ii = (temp[:, 2] >= t0) & (temp[:, 2] <= t1)  #index of items in this time period
        if sum(ii)>0:
            #ICUSTAY_ID, OUTCOMES, DEMOGRAPHICS
            reformat2[irow, 0] = (j/timestep)+1    # 'bloc' = timestep (1,2,3...)
            reformat2[irow, 1] = icustayid         # icustay_ID
            reformat2[irow, 2] = 3600*j+ beg       # t0 = lower limit of time window
            reformat2[irow, 3:11] = dem            # demographics and outcomes
            
            #CHARTEVENTS and LAB VALUES (+ includes empty cols for shock index and P/F)
            value = temp[ii]  # Records all values in this timestep
                
            if sum(ii) == 1:   # if only 1 row of values at this timestep
                reformat2[irow, 11:78] = value[:, 3:]
            else:
                reformat2[irow, 11:78] = np.nanmean(value[:, 3:], axis=0)  # mean of all available values
        
            # VASOPRESSORS
            # for CV: dose at timestamps.
            # for MV: 4 possibles cases, each one needing a different way to compute the dose of VP actually administered:
            #----t0---start----end-----t1----
            #----start---t0----end----t1----
            #-----t0---start---t1---end
            #----start---t0----t1---end----
            # MetaVision
            v = ((endv >= t0) & (endv <= t1)) | ((startv >= t0) & (endv<=t1)) | ((startv >= t0) & (startv <= t1))| ((startv <= t0) & (endv>=t1))
            # CareVue
            v2 = vaso2[(vaso2['charttime'] >= t0) & (vaso2['charttime'] <= t1)]['rate_std'].values
            if len(list(ratev[v]) +  list(v2)) > 0:
                v1 = np.nanmedian(list(ratev[v]) +  list(v2))
                v2 = np.nanmax(list(ratev[v]) + list(v2))
            else:
                v1 = np.nan
                v2 = np.nan
            
            if (~np.isnan(v1)) and (~np.isnan(v2)):
                reformat2[irow, 78] = v1    #median of dose of VP
                reformat2[irow, 79] = v2    #max dose of VP
        
            # INPUT FLUID
            # Input from MV (4 ways to compute)
            infu = np.nansum(rate*(endt-startt)*((endt <= t1) & (startt >= t0))/3600 
                                          + rate*(endt-t0)*((startt <= t0) & (endt <= t1) & (endt >= t0))/3600 
                                          + rate*(t1-startt)*((startt >= t0) & (endt >= t1) & (startt <= t1))/3600 
                                          + rate*(t1-t0)*((endt >=t1) & (startt <= t0))/3600)
            # All boluses received during this timestep, from inputMV (need to check rate is NaN) and inputCV (simpler):
            bolus = np.nansum(input[(np.isnan(input[:, 5])) & (input[:, 1] >= t0) & (input[:, 1] <= t1), 6]) + np.nansum(input2[(input2[:, 1] >= t0) & (input2[:, 1] <= t1), 4])
            # Cumulate all fluid given
            totvol = np.nansum([totvol, infu, bolus])
            reformat2[irow, 80] = totvol    #total fluid given
            reformat2[irow, 81] = np.nansum([infu, bolus])  #fluid given at this step
            
            # Urine Output
            UOnow = np.nansum(output[(output['charttime'] >= t0) & (output['charttime'] <= t1)]['value'].values)
            UOtot = np.nansum([UOtot, UOnow])
            reformat2[irow, 82] = UOtot    # Total Urine Output
            reformat2[irow, 83] = np.nansum(UOnow)  # Urine Output at this step

            #CUMULATED BALANCE
            reformat2[irow, 84] = totvol - UOtot

            #clinical note
            text_note = note1[(note1['charttime'] >= t0) & (note1['charttime'] <= t1)]['text'].values
            if len(list(text_note)) > 0:
                text_note = "".join(list(text_note))
                reformat2[irow, 85] = text_note
            else:
                # print("no note data: ", text_note)
                pass

            irow += 1

In [ ]:
reformat2 = np.delete(reformat2, range(irow, len(reformat2)) ,axis=0) 

# 13. CONVERT TO TABLE AND DELETE VARIABLES WITH EXCESSIVE MISSINGNESS

In [ ]:
print('FULL ICU -- CONVERTING TO TABLE AND DELETE VARIABLES WITH EXCESSIVE MISSINGNESS')
dataheaders = ['Height_cm', 'Weight_kg', 'GCS','RASS','HR', 'SysBP', 'MeanBP', 'DiaBP',	'RR', 'SpO2', 'Temp_C', 'Temp_F', 'CVP', 'PAPsys', 'PAPmean', 'PAPdia', 'CI', 
'SVR', 'Interface', 'FiO2_100', 'FiO2_1', 'O2flow', 'PEEP', 'TidalVolume', 'MinuteVentil', 'PAWmean', 'PAWpeak', 'PAWplateau', 'Potassium', 'Sodium',
'Chloride', 'Glucose', 'BUN', 'Creatinine', 'Magnesium', 'Calcium', 'Ionised_Ca', 'CO2_mEqL', 'SGOT', 'SGPT', 'Total_bili', 'Direct_bili', 'Total_protein',
'Albumin', 'Troponin', 'CRP', 'Hb', 'Ht', 'RBC_count', 'WBC_count', 'Platelets_count', 'PTT', 'PT', 'ACT', 'INR', 'Arterial_pH', 'paO2', 'paCO2',
'Arterial_BE', 'Arterial_lactate', 'HCO3', 'ETCO2', 'SvO2', 'mechvent', 'extubated', 'Shock_Index', 'PaO2_FiO2']

dataheaders = ['bloc','icustayid','charttime','gender','age','elixhauser','re_admission', 'died_in_hosp', 'died_within_48h_of_out_time','mortality_90d','delay_end_of_record_and_discharge_or_death'] + \
    dataheaders + ['median_dose_vaso','max_dose_vaso','input_total','input_4hourly','output_total','output_4hourly','cumulated_balance', 'clinical_note']

In [ ]:
reformat2t = pd.DataFrame(reformat2, columns=dataheaders)

In [ ]:
miss = np.sum(np.isnan(reformat2[:,:85].astype(float)), axis=0)/reformat2.shape[0]

In [ ]:
# If values have less than 70% missing values (over 30% of values present): I keep them
reformat3 = reformat2[:,[True]*11 + (miss[11:74] < 0.7).tolist() + [True]*12]
reformat3t = pd.DataFrame(reformat3, columns= reformat2t.columns[[True]*11 + (miss[11:74] < 0.7).tolist() + [True]*12])

# 14. HANDLING OF MISSING VALUES  &  CREATE REFORMAT4T

In [ ]:
# Do linear interpolation where missingness is low (kNN imputation doesnt work if all rows have missing values)
print('Full ICU -- Doing linear interpolation where missingness is low (kNN imputation doesnt work if all rows have missing values)')
miss = np.sum(np.isnan(reformat3[:,:66].astype(float)), axis=0)/reformat3.shape[0]

In [ ]:
ii = (miss>0) & (miss<0.05)  #less than 5% missingness
mechventcol = reformat3t.columns.tolist().index('mechvent')

for i in range(10,mechventcol): # Correct column by column
    if ii[i]==1:
        reformat3[:,i] = fixgaps(reformat3[:,i].astype(float))

In [ ]:
reformat3t[reformat3t.columns[10:mechventcol]] = reformat3[:,10:mechventcol]

In [ ]:
# KNN IMPUTATION -  Done on chunks of 10K records.
print('Full ICU -- KNN imputation')

reformat3t_cols = reformat3t.columns.tolist()
mechventcol = reformat3t_cols.index('mechvent')
ref = np.copy(reformat3[:,11:mechventcol])  #columns of interest

bar_knn = pyprind.ProgBar(len(range(0,reformat3.shape[0],9999)))
for i in range(0,reformat3.shape[0],9999):   #dataset divided in 10K rows chunks (otherwise too large)
    bar_knn.update()
    ref[i:i+9999,:] = KNN(k=1).fit_transform(ref[i:i+9999,:])

reformat3t[reformat3t_cols[11:mechventcol]] = ref 

reformat4t = reformat3t.copy()
reformat4 = reformat4t.values

# 15. COMPUTE SOME DERIVED VARIABLES: P/F, Shock Index, SOFA, SIRS...

In [ ]:
reformat4t = reformat3t.copy()
reformat4 = reformat4t.values

In [ ]:
print('FULL ICU -- COMPUTING SOME DERIVED VARIABLES: P/F, Shock Index, SOFA, SIRS...')
# CORRECT GENDER
reformat4t['gender'] = reformat4t['gender'] - 1

# CORRECT AGE > 200 yo
ii = reformat4t['age'] > 150*365.25
reformat4t.loc[ii,'age'] = 91.4*365.25

# FIX MECHVENT
reformat4t['mechvent'].fillna(0, inplace=True)
reformat4t.loc[reformat4t['mechvent'] > 0, 'mechvent'] = 1

# FIX Elixhauser missing values
reformat4t['elixhauser'].loc[np.isnan(reformat4t['elixhauser'].astype(float))] = np.nanmedian(reformat4t['elixhauser'])   #use the median value / only a few missing data points 

# Vasopressors / no NAN
reformat4t['median_dose_vaso'].fillna(0, inplace=True)
reformat4t['max_dose_vaso'].fillna(0, inplace=True)

# clinical note / no NAN
reformat4t['clinical_note'].fillna(0, inplace=True)

# Recompute P/F with no missing values...
reformat4t['PaO2_FiO2'] = reformat4t['paO2']/reformat4t['FiO2_1']

# Recompute SHOCK INDEX without NAN and INF
reformat4t['Shock_Index'] = reformat4t['HR'].astype(float)/reformat4t['SysBP'].astype(float)

reformat4t.loc[np.isinf(reformat4t['Shock_Index']), 'Shock_Index'] = np.nan

d = np.nanmean(reformat4t['Shock_Index'])
reformat4t['Shock_Index'].fillna(d, inplace=True)

In [ ]:
# SOFA - at each timepoint we need (in this order):  
# P/F,  MV,  PLT,  TOT_BILI,  MAP,  NORAD(max),  GCS,  CR,  UO
s = reformat4t[['PaO2_FiO2', 'Platelets_count', 'Total_bili', 'MeanBP', 'max_dose_vaso', 'GCS', 'Creatinine', 'output_4hourly']].values
p = np.arange(5)
s1=np.array([s[:,0]>400, (s[:, 0]>=300) & (s[:, 0]<400), (s[:, 0]>=200) & (s[:, 0]<300), (s[:, 0]>=100) & (s[:, 0]<200), s[:, 0]<100 ])   #count of points for all 6 criteria of sofa
s2=np.array([s[:,1]>150, (s[:, 1]>=100) & (s[:, 1]<150), (s[:, 1]>=50) & (s[:, 1]<100), (s[:, 1]>=20) & (s[:, 1]<50), s[:, 1]<20 ])
s3=np.array([s[:, 2]<1.2, (s[:, 2]>=1.2) & (s[:, 2]<2), (s[:, 2]>=2) & (s[:, 2]<6), (s[:, 2]>=6) & (s[:, 2]<12), s[:, 2]>12 ])
s4=np.array([s[:, 3]>=70, (s[:, 3]<70) & (s[:, 3]>=65), (s[:, 3]<65), (s[:, 4]>0) & (s[:, 4]<=0.1), s[:, 4]>0.1 ])
s5=np.array([s[:, 5]>14, (s[:, 5]>12) & (s[:, 5]<=14), (s[:, 5]>9) & (s[:, 5]<=12), (s[:, 5]>5) & (s[:, 5]<=9), s[:, 5]<=5])
s6=np.array([s[:, 6]<1.2, (s[:, 6]>=1.2) & (s[:, 6]<2), (s[:, 6]>=2) & (s[:, 6]<3.5), ((s[:, 6]>=3.5) & (s[:, 6]<5))|(s[:, 7]<84), (s[:, 6]>5)|(s[:, 7]<34)])

num_columns = reformat4t.shape[1]   #nr of variables in data
newcols_reformat4 = np.zeros((reformat4t.shape[0],7)) 
for i in range(reformat4t.shape[0]):
    t = max(p[s1[:, i]], default=0) + max(p[s2[:, i]], default=0) + max(p[s3[:, i]], default=0) + max(p[s4[:, i]], default=0) + max(p[s5[:, i]], default=0) + max(p[s6[:, i]], default=0)  #SUM OF ALL 6 CRITERIA
    if t > 0:
        newcols_reformat4[i, :] = [max(p[s1[:, i]], default=0), max(p[s2[:, i]], default=0), max(p[s3[:, i]], default=0), max(p[s4[:, i]], default=0), max(p[s5[:, i]], default=0), max(p[s6[:, i]], default=0), t]


In [ ]:
# SIRS - at each timepoint |  need: temp HR RR PaCO2 WBC 
s = reformat4t[['Temp_C', 'HR', 'RR', 'paCO2', 'WBC_count']].values

s1=np.array([(s[:, 0]>=38) | (s[:,0]<=36)])   # Count of points for all criteria of SIRS
s2=np.array([s[:, 1]>90])
s3=np.array([(s[:, 2]>=20) | (s[:, 3]<=32)])
s4=np.array([(s[:, 4]>=12) | (s[:, 4]<4)])
newcols_sirs = (1*s1) + (1*s2) + (1*s3) + (1*s4)

In [ ]:
# Adds 2 cols for SOFA and SIRS
# Records values
reformat4t['SOFA'] = newcols_reformat4[:,-1]
reformat4t['SIRS'] = newcols_sirs[0]

# 16. EXCLUSION OF SOME PATIENTS 

In [ ]:
# Check for patients with extreme UO = outliers = to be deleted (>40 litres of UO per 4h!!)
a = reformat4t['output_4hourly'] > 12000 
i = reformat4t[a]['icustayid'].unique() 
i = reformat4t['icustayid'].isin(i) 
reformat4t.drop(reformat4t.index[i], inplace=True) 

# Some have bili = 999999
a = reformat4t['Total_bili'] > 10000 
i = reformat4t[a]['icustayid'].unique()
i = reformat4t['icustayid'].isin(i) 
reformat4t.drop(reformat4t.index[i], inplace=True)

# Check for patients with extreme INTAKE = outliers = to be deleted (>10 litres of intake per 4h!!)
a = reformat4t['input_4hourly'] > 10000 
i = reformat4t[a]['icustayid'].unique()
i = reformat4t['icustayid'].isin(i) 
reformat4t.drop(reformat4t.index[i], inplace=True)

In [ ]:
## Save a copy as a precaution.
# import pickle
# reformat4t.to_csv("./preprocessed_files_with_note/reformat4t.csv", index=False)
# datasss = {"dict": qstime, "array": icustayidlist}
# with open("./preprocessed_files_with_note/data.pkl", "wb") as f:
#     pickle.dump(datasss, f)

In [ ]:
# with open("./preprocessed_files_with_note/data.pkl", "rb") as f:
#     datasss = pickle.load(f)

In [ ]:
from tqdm import tqdm
# Exclude early deaths that may reflect withdrawal of treatment.
print('Full ICU -- Excluding early deaths from possible withdrawals')
# Stats per patient
q = reformat4t['bloc']==1

num_of_trials = len(reformat4t['icustayid'].unique()) 
a = reformat4t[['icustayid', 'mortality_90d', 'max_dose_vaso', 'SOFA']].values
a = pd.DataFrame(a, columns = ['id', 'mortality_90d', 'vaso', 'sofa']) 
d = a.groupby(['id']).max() 
d_count = a.groupby(['id']).count()
# Find the patients who match the Sepsis 3 criteria
e = np.zeros(num_of_trials)
for i in tqdm(range(num_of_trials)):
    if d['mortality_90d'].iloc[i] == 1:
        ii = (reformat4t['icustayid'] == d.index[i]) & (reformat4t['bloc'] == d_count.iloc[i]['mortality_90d'])  #last row for this patient
        e[i] = np.sum((reformat4t['max_dose_vaso'][ii] == 0) & (d['vaso'].iloc[i] > 0.3) & (reformat4t['SOFA'][ii] >= d['sofa'].iloc[i]/2)) > 0
r = d.index[(e == 1) & (d_count['mortality_90d'] < 20)] # ids to be removed
ii = reformat4t['icustayid'].isin(r) 


In [ ]:
reformat4t = reformat4t.loc[~ii]

In [ ]:
# Exclude patients who died in ICU during data collection period
print('Full ICU -- excluding patients who died in ICU during data collection period')
ii = (reformat4t['bloc'] == 1) & (reformat4t['died_within_48h_of_out_time'] == 1) & (reformat4t['delay_end_of_record_and_discharge_or_death'] < 24)
ii = reformat4t['icustayid'][ii][reformat4t['icustayid'][ii].isin(icustayidlist)]
ii = reformat4t['icustayid'].isin(ii)
reformat4t = reformat4t.loc[~ii] 

# 17. CREATE SEPSIS COHORT FROM ALL ICU PATIENTS EXTRACTED

In [ ]:
print('Creating sepsis cohort')
# Create array with 1 row per icu admission
# Keep only patients with flagged sepsis (max sofa during time period of interest >= 2)
# Assumed baseline SOFA is zero
sepsis = np.zeros((30000,5)) #NOTE: For other cohorts, this size may have to be changed
irow = 0

bar_cohort = pyprind.ProgBar(len(icustayidlist))
for icustayid in icustayidlist:
    bar_cohort.update()
    ii = reformat4t['icustayid'] == icustayid 
    if sum(ii) > 0:
        sofa = reformat4t['SOFA'][ii]
        sirs = reformat4t['SIRS'][ii]
        sepsis[irow, 0] = icustayid
        sepsis[irow, 1] = reformat4t['mortality_90d'][ii].iloc[0] # 90-day mortality
        sepsis[irow, 2] = np.max(sofa)
        sepsis[irow, 3] = np.max(sirs)
        sepsis[irow, 4] = qstime[icustayid][0]   #time of onset of sepsis #icustayid-1 not done to keep it consistent with earlier verified use of qstime and 0 added as onset of sepsis index.
        irow += 1


In [ ]:
sepsis = np.delete(sepsis, range(irow, len(sepsis)) ,axis=0) # Remove extra rows
sepsis = pd.DataFrame(sepsis, columns=['icustayid', 'morta_90d', 'max_sofa', 'max_sirs', 'sepsis_time'])

In [ ]:
# Delete all non-septic patients
ii = sepsis['max_sofa'] < 2
sepsis = sepsis[~ii]
# Final count of patients included
print('Final patient count:', sepsis.shape[0])  

In [ ]:
# Save cohort
sepsis.to_csv('./preprocessed_files_with_note/new_sepsis_mimiciii.csv', index=False) 

# 18. sepsis 최종 환자 id기준으로 동일한 전처리 다시 수행

In [ ]:
#sepsis = pd.read_csv("./preprocessed_files_with_note/new_sepsis_mimiciii.csv")

In [ ]:
########################################################################
#           INITIAL REFORMAT WITH CHARTEVENTS, LABS AND MECHVENT
########################################################################
#gives an array with all unique charttime (1 per row) and all items in columns.
################## IMPORTANT !!!!!!!!!!!!!!!!!!
# Here we use -24 -> +48 to define the MDP
print('Sepsis Cohort -- Making an array with all unique charttime (1 per row) and all items in columns.')
reformat = np.nan*np.ones((2000000,69))  # Final table: add one column for the presumed onset time.
qstime = dict()
winb4 = 25   #lower limit for inclusion of data (24h before time flag)
winaft = 49  # upper limit (48h after)
irow = 0  #recording row for summary table
bar = pyprind.ProgBar(sepsis.shape[0]+1)
for icustayidrow in range(1, sepsis.shape[0]+1):
    bar.update()
    qst = sepsis['sepsis_time'].iloc[icustayidrow-1] #;%,3); %flag for presumed infection
    icustayid = int(sepsis['icustayid'].iloc[icustayidrow-1])
    qstime[icustayid] = np.zeros(4)
    # CHARTEVENTS
    if (icustayid-200000) < 10000:
        temp=ce010
    elif (icustayid-200000) < 20000:
        temp=ce1020
    elif (icustayid-200000) < 30000:
        temp=ce2030
    elif (icustayid-200000) < 40000:
        temp=ce3040
    elif (icustayid-200000) < 50000:
        temp=ce4050
    elif (icustayid-200000) < 60000:
        temp=ce5060
    elif (icustayid-200000) < 70000:
        temp=ce6070
    elif (icustayid-200000) < 80000:
        temp=ce7080
    elif (icustayid-200000) < 90000:
        temp=ce8090
    else:
        temp=ce90100
    temp = temp[temp['icustay_id'] == icustayid]

    ii = (temp['charttime'] >= qst - (winb4+4)*3600) & (temp['charttime'] <= qst + (winaft+4)*3600) #time period of interest -4h and +4h
    temp = temp.loc[ii]   #only time period of interest

    # LAB EVENTS
    ii = labU['icustay_id'] == icustayid
    temp2 = labU.loc[ii]
    ii = (temp2['charttime'] >= qst - (winb4+4)*3600) & (temp2['charttime'] <= qst + (winaft+4)*3600) #time period of interest -4h and +4h
    temp2 = temp2.loc[ii]   #only time period of interest

    # Mech Vent + ?extubated
    ii = MV['icustay_id'] == icustayid
    temp3 = MV.loc[ii]
    ii = (temp3['charttime'] >= qst - (winb4+4)*3600) & (temp3['charttime'] <= qst + (winaft+4)*3600) # Time period of interest -4h and +4h
    temp3 = temp3.loc[ii]   #only time period of interest
    
    t = np.unique(pd.concat([temp['charttime'], temp2['charttime'], temp3['charttime']], ignore_index=True).values) # List of unique timestamps from all 3 sources / sorted in ascending order

    if len(t) > 0:
        for i in range(len(t)):
            # CHARTEVENTS
            ii = temp['charttime'] == t[i]
            col = temp.loc[ii, 'itemid']
            value = temp.loc[ii, 'valuenum']
            reformat[irow, 0] = i+1 # Timestep  
            reformat[irow, 1] = icustayid
            reformat[irow, 2] = t[i] # Charttime
            reformat[irow, 3] = qst  # Store the presumed onset time
            reformat[irow, 3+col.astype(int).values] = value.values # Store available values

            #LAB VALUES
            ii = temp2['charttime'] == t[i]
            col = temp2.loc[ii, 'itemid']
            value = temp2.loc[ii, 'valuenum']
            reformat[irow,31+col.astype(int).values] = value.values  #store available values
        
            #MV  
            ii = temp3['charttime'] == t[i]
            if np.nansum(ii) > 0:
                col = temp3.loc[ii, 'mechvent']
                value = temp3.loc[ii, 'extubated']
                reformat[irow, 67] = col.values[0] # Store available values
                reformat[irow, 68] = value.values[0] # Store available values
            else:
                reformat[irow, 67]= np.nan
                reformat[irow, 68]= np.nan
            irow += 1

        qstime[icustayid][0] = qst # Flag for presumed infection / this is time of sepsis if SOFA >=2 for this patient
        # WE SAVE FIRST and LAST TIMESTAMPS, in QSTIME, for each ICUSTAYID
        qstime[icustayid][1] = t[0]   # First timestamp
        qstime[icustayid][2] = t[-1]  # Last timestamp
        qstime[icustayid][3] = demog.loc[demog['icustay_id'] == icustayid, 'dischtime'].values[0] # Discharge time

reformat = np.delete(reformat, range(irow, len(reformat)) ,axis=0)  # Remove unused rows

In [ ]:
########################################################################
#                                   OUTLIERS 
########################################################################
print('Sepsis Cohort -- Handling outliers')

# Weight
reformat = deloutabove(reformat, 5, 300)

# Heart Rate
reformat = deloutabove(reformat, 8, 250)

# Blood Pressure
reformat = deloutabove(reformat, 9, 300)
reformat = deloutbelow(reformat, 10, 0)
reformat = deloutabove(reformat, 11, 200)
reformat = deloutbelow(reformat, 11, 0)
reformat = deloutabove(reformat, 11, 200)

# Respiratory Rate
reformat = deloutabove(reformat, 12, 80) 

# SpO2
reformat = deloutabove(reformat, 13, 150)
reformat[reformat[:, 13]>100, 13] = 100 
reformat = deloutbelow(reformat, 13, 50) 

# Temperature
reformat[(reformat[:, 14] > 90) & (np.isnan(reformat[:, 15])), 15] = reformat[(reformat[:, 14] > 90) & (np.isnan(reformat[:, 15])), 14]
reformat = deloutabove(reformat, 14, 90) 
reformat = deloutbelow(reformat, 14, 25) 

# interface / is in col 23
# FiO2
reformat = deloutabove(reformat, 23, 100)
reformat[reformat[:, 23] < 1 , 23] = reformat[reformat[:, 23] < 1 , 23]*100
reformat = deloutbelow(reformat, 23, 20) 
reformat = deloutabove(reformat, 24, 1.5)

# O2 FLOW
reformat = deloutabove(reformat, 25, 70)

# PEEP
reformat=deloutbelow(reformat, 26, 0)
reformat=deloutabove(reformat, 26, 40)

# Total Volume
reformat=deloutabove(reformat, 27, 1800)

# Mean Volume
reformat=deloutabove(reformat, 28, 50)

# Potassium
reformat=deloutbelow(reformat, 32, 1)
reformat=deloutabove(reformat, 32, 15)

# Sodium
reformat=deloutbelow(reformat, 33, 95)
reformat=deloutabove(reformat, 33, 178)

# Chloride
reformat=deloutbelow(reformat, 34, 70)
reformat=deloutabove(reformat, 34, 150)

# Glucose
reformat=deloutbelow(reformat, 35, 1)
reformat=deloutabove(reformat, 35, 1000) 

# Creatinine
reformat=deloutabove(reformat, 37, 150)

# Magnesium
reformat=deloutabove(reformat, 38, 10)

# Calcium
reformat=deloutabove(reformat, 39, 20)

# Ionized Calcium
reformat=deloutabove(reformat, 40, 5)

# CO2
reformat=deloutabove(reformat, 41, 120)

# SGPT/SGOT
reformat=deloutabove(reformat, 42, 10000)
reformat=deloutabove(reformat, 43, 10000)

# Hb/Ht
reformat=deloutabove(reformat, 50, 20)
reformat=deloutabove(reformat, 51, 65)

# White Blood Cells
reformat=deloutabove(reformat, 53, 500)

# Platelets
reformat=deloutabove(reformat, 54, 2000)

# INR
reformat=deloutabove(reformat, 58, 20)

# pH
reformat=deloutbelow(reformat, 59, 6.7)
reformat=deloutabove(reformat, 59, 8) 

# pO2
reformat=deloutabove(reformat, 60, 700)

# pCO2
reformat=deloutabove(reformat, 61, 200)

# Base Excess
reformat=deloutbelow(reformat, 62, -50)

# Lactate
reformat=deloutabove(reformat, 63, 30)

In [ ]:
# More data manipulation / imputation from existing values
# Estimate GCS from RASS - data from Wesley JAMA 2003
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] >= 0), 6] = 15
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -1), 6] = 14
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -2), 6] = 12
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -3), 6] = 11
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -4), 6] = 6
reformat[(np.isnan(reformat[:, 6])) & (reformat[:, 7] == -5), 6] = 3

# FiO2
reformat[(~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 24])), 24] = reformat[(~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 24])), 23] / 100
reformat[(~np.isnan(reformat[:, 24])) & (np.isnan(reformat[:, 23])), 23] = reformat[(~np.isnan(reformat[:, 24])) & (np.isnan(reformat[:, 23])), 24] * 100


In [ ]:
# ESTIMATE FiO2 /// with use of interface / device (cannula, mask, ventilator....)
print('Sepsis Cohort -- Doing sample and hold')
reformatsah = SAH(reformat, sample_and_hold, adjust=1) # do SAH first to handle this task, setting `adjust` to 1 since we added another column

# NO FiO2, YES O2 flow, no interface OR cannula
ii = np.where((np.isnan(reformatsah[:,23])) & (~np.isnan(reformatsah[:,25])) & ((reformatsah[:,22] == 0) | (reformatsah[:,22] == 2)))[0] #As np.where returns a tuple
reformat[ii[reformatsah[ii, 25] <= 15], 23] = 70
reformat[ii[reformatsah[ii, 25] <= 12], 23] = 62
reformat[ii[reformatsah[ii, 25] <= 10], 23] = 55
reformat[ii[reformatsah[ii, 25] <= 8], 23]  = 50
reformat[ii[reformatsah[ii, 25] <= 6], 23]  = 44
reformat[ii[reformatsah[ii, 25] <= 5], 23]  = 40
reformat[ii[reformatsah[ii, 25] <= 4], 23]  = 36
reformat[ii[reformatsah[ii, 25] <= 3], 23]  = 32
reformat[ii[reformatsah[ii, 25] <= 2], 23]  = 28
reformat[ii[reformatsah[ii, 25] <= 1], 23]  = 24

# NO FiO2, NO O2 flow, no interface OR cannula
ii = np.where((np.isnan(reformatsah[:, 23])) & np.isnan(reformatsah[:, 25]) & ((reformatsah[:, 22] == 0) | (reformatsah[:, 22] == 2)))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 23] = 21

# NO FiO2, YES O2 flow, face mask OR.... OR ventilator (assume it's face mask)
ii = np.where((np.isnan(reformatsah[:,23])) & (~np.isnan(reformatsah[:,25])) & 
((reformatsah[:, 22]==1) | (reformatsah[:, 22]==3) | (reformatsah[:, 22]==4) | (reformatsah[:, 22]==5) | (reformatsah[:, 22]==6) | (reformatsah[:, 22]==9) | (reformatsah[:, 22]==10)))[0]
reformat[ii[reformatsah[ii, 25]<=15], 23] = 75
reformat[ii[reformatsah[ii, 25]<=12], 23] = 69
reformat[ii[reformatsah[ii, 25]<=10], 23] = 66
reformat[ii[reformatsah[ii, 25]<=8], 23]  = 58
reformat[ii[reformatsah[ii, 25]<=6], 23]  = 40
reformat[ii[reformatsah[ii, 25]<=4], 23]  = 36

# NO FiO2, NO O2 flow, face mask OR ....OR ventilator
ii = np.where(np.isnan(reformatsah[:, 23]) & np.isnan(reformatsah[:, 25]) & ((reformatsah[:, 22] == 1) | (reformatsah[:, 22] == 3) | 
(reformatsah[:, 22] == 4) | (reformatsah[:, 22] == 5) | (reformatsah[:, 22] == 6) | (reformatsah[:, 22] == 9) | (reformatsah[:, 22] == 10)))[0]  # No FiO2 given and O2 flow given, no interface OR cannula
reformat[ii, 23] = np.nan

# NO FiO2, YES O2 flow, Non rebreather mask
ii = np.where(np.isnan(reformatsah[:, 23]) & (~np.isnan(reformatsah[:, 25])) & (reformatsah[:, 22] == 7))[0]
reformat[ii[reformatsah[ii, 25] >= 10], 23] = 90
reformat[ii[reformatsah[ii, 25] >= 15], 23] = 100
reformat[ii[reformatsah[ii, 25] < 10], 23]  = 80
reformat[ii[reformatsah[ii, 25] <= 8], 23]  = 70
reformat[ii[reformatsah[ii, 25] <= 6], 23]  = 60

# NO FiO2, NO O2 flow, NRM
ii= np.where(np.isnan(reformatsah[:, 23]) & np.isnan(reformatsah[:, 25]) & (reformatsah[:, 22]==7))[0]  #no fio2 given and o2flow given, no interface OR cannula
reformat[ii, 23] = np.nan

# Update Fi02 columns again
ii = (~np.isnan(reformat[:, 23])) & (np.isnan(reformat[:, 24]))
reformat[ii, 24] = reformat[ii, 23]/100
ii = (~np.isnan(reformat[:, 24])) & (np.isnan(reformat[:, 24]))
reformat[ii, 23] = reformat[ii, 24]*100

# BLOOD PRESSURE
ii = (~np.isnan(reformat[:, 9])) & (~np.isnan(reformat[:, 10])) & np.isnan(reformat[:, 11])
reformat[ii, 11] = (3*reformat[ii, 10] - reformat[ii, 9])/2
ii = (~np.isnan(reformat[:, 9])) & (~np.isnan(reformat[:, 11])) & np.isnan(reformat[:, 10])
reformat[ii, 10] = (reformat[ii, 9] + 2*reformat[ii, 11])/3
ii = (~np.isnan(reformat[:, 10])) & (~np.isnan(reformat[:, 11])) & np.isnan(reformat[:, 9])
reformat[ii, 9] = 3*reformat[ii, 10] - 2*reformat[ii, 11]

# TEMPERATURE
# some values recorded in the wrong column
ii = (reformat[:, 15] > 25) & (reformat[:, 15] < 45) #tempF close to 37deg??!
reformat[ii, 14] = reformat[ii, 15]
reformat[ii, 15] = np.nan
ii = reformat[:, 14] >70  # TempC > 70, likely recorded in Farenheit
reformat[ii, 15] = reformat[ii, 14]
reformat[ii, 14] = np.nan
ii = (~np.isnan(reformat[:, 14])) & np.isnan(reformat[:, 15])
reformat[ii, 15] = reformat[ii, 14] * 1.8 + 32
ii = (~np.isnan(reformat[:, 14])) & np.isnan(reformat[:, 13])
reformat[ii, 14] = (reformat[ii, 15] - 32)/1.8

# Hb/Ht
ii = (~np.isnan(reformat[:, 50])) & np.isnan(reformat[:,51])
reformat[ii, 51] = (reformat[ii, 50] * 2.862) + 1.216
ii = (~np.isnan(reformat[:, 51])) & np.isnan(reformat[:,50])
reformat[ii, 50] = (reformat[ii, 51] - 1.216)/2.862

# BILIRUBIN
ii = (~np.isnan(reformat[:, 44])) & np.isnan(reformat[:,45])
reformat[ii, 45] = (reformat[ii, 44]*0.6934)-0.1752
ii = (~np.isnan(reformat[:, 45])) & np.isnan(reformat[:,44])
reformat[ii, 44] = (reformat[ii, 45] + 0.1752)/0.6934

In [ ]:
########################################################################
#                      SAMPLE AND HOLD on RAW DATA
########################################################################
print('SEPSIS COHORT -- SAMPLE AND HOLD on RAW DATA')
reformat = SAH(reformat[:, 0:69], sample_and_hold, adjust=1)  # Setting `adjust` to 1 to account for the added column for `presumed_onset`
## Use adjust to exclude timestep, icustayid, charttime, and qst from SAH.
#######################################################################

In [ ]:
# reformat_copy = reformat

In [ ]:
#reformat = reformat_copy

In [ ]:
########################################################################
#                             DATA COMBINATION
########################################################################
print('Sepsis Cohort -- Data combination')
# WARNING: the time window of interest has been defined above (here -24 -> +48)! 
timestep = 4  # Resolution of timesteps, in hours
irow = 0   
icustayidlist = np.unique(reformat[:,1]).astype(np.int32)
reformat2 = np.nan*np.ones((reformat.shape[0], 87))  # Output array: add one column for notes.
reformat2 = reformat2.astype(object)

num_patients = len(icustayidlist)  # Number of patients
# Adding 2 empty cols for future shock index=HR/SBP and P/F
reformat = np.hstack([reformat, np.nan*np.ones((reformat.shape[0], 2))])

In [ ]:

bar = pyprind.ProgBar(num_patients)
for i in range(len(icustayidlist)):
    bar.update()
    icustayid = icustayidlist[i]  
    
    #CHARTEVENTS AND LAB VALUES
    temp = reformat[reformat[:,1] == icustayid,:]   # Subtable of interest
    beg = temp[0,2]   # Timestamp of first record
    
    #IV FLUID STUFF
    iv = inputMV['icustay_id'] == icustayid   # Rows of interest in inputMV
    input = inputMV[iv]    # Subset of interest
    iv = inputCV['icustay_id'] == icustayid   # Rows of interest in inputCV
    input2 = inputCV[iv]    # Subset of interest
    startt = input['starttime'] # Start of all infusions and boluses
    endt = input['endtime'] # End of all infusions and boluses
    rate = input['norm_rate_of_infusion']  # Rate of infusion (is NaN for boluses) || corrected for tonicity
    pread = inputpreadm[inputpreadm['icustay_id'] == icustayid]['inputpreadm'] # Preadmission volume

    input = input.values
    input2 = input2.values

    if len(pread) >0:             # Store the value, if available
        totvol = np.nansum(pread)
    else:
        totvol=0   # If not documented: it's zero

    # Compute volume of fluid given before start of record!!!
    t0 = 0
    t1 = beg
    # Input from MV (4 ways to compute)
    infu = np.nansum(rate*(endt-startt)*((endt<=t1) & (startt >= t0))/3600 + rate*(endt-t0)*((startt <= t0) & (endt <= t1) & (endt >= t0))/3600 +
                            rate*(t1-startt)*((startt >= t0) & (endt >=t1) & (startt <= t1))/3600 + rate*(t1-t0)*((endt >= t1) & (startt <= t0))/3600)
    # All boluses received during this timestep, from inputMV (need to check rate is NaN) and inputCV (simpler):
    bolus = np.nansum(input[np.isnan(input[:,5]) & (input[:,1] >= t0) & (input[:,1] <= t1),6]) + np.nansum(input2[(input2[:,1] >= t0) & (input2[:,1] <= t1),4])
    totvol = np.nansum([totvol,infu,bolus]) 
    
    #########################################################################################
    #VASOPRESSORS    
    iv = vasoMV['icustay_id'] == icustayid # rows of interest in vasoMV
    vaso1 = vasoMV[iv]    # subset of interest
    iv = vasoCV['icustay_id'] == icustayid   # rows of interest in vasoCV
    vaso2 = vasoCV[iv]    # subset of interest
    startv = vaso1['starttime'].values  # start of VP infusion
    endv = vaso1['endtime'].values      # end of VP infusions
    ratev = vaso1['rate_std'].values  # rate of VP infusion
    #########################################################################################
    #note
    noti = note['icustay_id'] == icustayid  # rows of interest in vasoMV
    note1 = note[noti]    # subset of interest

    
    # DEMOGRAPHICS / gender, age, elixhauser, re-admit, onset time, died in hosp?, died within 48h of out_time (likely in ICU or soon after),
    # died within 90d after admission?, Length of stay after obs window     
    demogi = demog['icustay_id'] == icustayid
    dem = np.array(
            list(demog.gender[demogi].values) +
            list(demog.age[demogi].values) +
            list(demog.elixhauser[demogi].values) +
            list(demog.re_admission[demogi].values) + # Using corrected readmission (within 30 days of previous discharge)
            [temp[0,3]] +                    # Adding presumed sepsis onset
            list(demog.morta_hosp[demogi].values) + 
            list(abs(demog.dod[demogi].values - demog.outtime[demogi].values) < (24*3600*2)) +
            list(demog.morta_90[demogi].values) + 
            [(qstime[icustayid][3] - qstime[icustayid][2])/3600]      # Length of time after last observation and discharge?
            )
  
    # URINE OUTPUT
    iu = UO['icustay_id'] == icustayid   #rows of interest in inputMV
    output = UO[iu]    #subset of interest
    pread = UOpreadm[UOpreadm['icustay_id'] == icustayid-200000]['value'].values #preadmission UO ????????????????? Why no + 200000 for icustayid here?
    if len(pread) > 0:     #store the value, if available
        UOtot = np.nansum(pread)
    else:
        UOtot = 0
    # adding the volume of urine produced before start of recording!    
    UOnow = np.nansum(output[(output['charttime']>=t0) & (output['charttime'] <= t1)]['value'].values) #t0 and t1 defined above
    UOtot = np.nansum([UOtot, UOnow])
    
    # Loop over the relevant times for each patient where information is recorded
    for j in range(0, 79, timestep): # 0:timestep:79 % -28 until +52 = 80 hours in total (4 hours buffer on either side of our desired window)
        t0 = 3600*j + beg   # left limit of time window
        t1 = 3600*(j + timestep) + beg   # right limit of time window
        ii = (temp[:,2] >= t0) & (temp[:,2] <= t1)  # index of items in this time period
        if sum(ii)>0:
            # ICUSTAY_ID, OUTCOMES, DEMOGRAPHICS
            reformat2[irow,0] = (j/timestep)+1    # 'bloc' = timestep (1,2,3...)
            reformat2[irow,1] = icustayid         # icustay_ID
            reformat2[irow,2] = 3600*j+ beg       # t0 = lower limit of time window
            reformat2[irow,3:12] = dem            # demographics and outcomes  
            
            # CHARTEVENTS and LAB VALUES (+ includes empty columns for shock index and P/F)
            value = temp[ii]  #records all values in this timestep
                
            if sum(ii) == 1:   # if only 1 row of values at this timestep
                reformat2[irow,12:79] = value[:,4:]    
            else:
                reformat2[irow,12:79] = np.nanmean(value[:,4:], axis=0) # mean of all available values   

            #VASOPRESSORS
            # for CV: dose at timestamps.
            # for MV: 4 possibles cases, each one needing a different way to compute the dose of VP actually administered:
            #----t0---start----end-----t1----
            #----start---t0----end----t1----
            #-----t0---start---t1---end
            #----start---t0----t1---end----
            #MV
            v = ((endv >= t0) & (endv <= t1)) | ((startv >= t0) & (endv<=t1)) | ((startv >= t0) & (startv <= t1))| ((startv <= t0) & (endv>=t1))
            #CV
            v2 = vaso2[(vaso2['charttime'] >= t0) & (vaso2['charttime'] <= t1)]['rate_std'].values
            if len(list(ratev[v]) +  list(v2)) > 0:
                v1 = np.nanmedian(list(ratev[v]) +  list(v2))
                v2 = np.nanmax(list(ratev[v]) + list(v2))
            else:
                v1 = np.nan
                v2 = np.nan
            if (~np.isnan(v1)) and (~np.isnan(v2)):
                reformat2[irow,79] = v1    #median of dose of VP  
                reformat2[irow,80] = v2    #max dose of VP  

            #INPUT FLUID
            #input from MV (4 ways to compute)
            infu = np.nansum(rate*(endt-startt)*((endt <= t1) & (startt >= t0))/3600 
                                          + rate*(endt-t0)*((startt <= t0) & (endt <= t1) & (endt >= t0))/3600 
                                          + rate*(t1-startt)*((startt >= t0) & (endt >= t1) & (startt <= t1))/3600 
                                          + rate*(t1-t0)*((endt >=t1) & (startt <= t0))/3600)
            #all boluses received during this timestep, from inputMV (need to check rate is NaN) and inputCV (simpler):
            bolus = np.nansum(input[(np.isnan(input[:,5])) & (input[:,1] >= t0) & (input[:,1] <= t1),6]) + np.nansum(input2[(input2[:,1] >= t0) & (input2[:,1] <= t1),4])
            #sum fluid given
            totvol = np.nansum([totvol, infu, bolus])
            reformat2[irow,81] = totvol    #total fluid given
            reformat2[irow,82] = np.nansum([infu, bolus])  #fluid given at this step
            
            #UO
            UOnow = np.nansum(output[(output['charttime'] >= t0) & (output['charttime'] <= t1)]['value'].values)
            UOtot = np.nansum([UOtot, UOnow])
            reformat2[irow,83] = UOtot    #total UO
            reformat2[irow,84] = np.nansum(UOnow)   #UO at this step

            #CUMULATED BALANCE
            reformat2[irow,85] = totvol - UOtot    #cumulated balance

            #clinical note
            text_note = note1[(note1['charttime'] >= t0) & (note1['charttime'] <= t1)]['text'].values
            if len(list(text_note)) > 0:
                text_note = "".join(list(text_note))
                reformat2[irow, 86] = text_note
            else:
                # print("no note data: ", text_note)
                pass

            irow += 1

In [ ]:
reformat2 = np.delete(reformat2, range(irow, len(reformat2)) ,axis=0)

In [ ]:
reformat2.shape

In [ ]:
#########################################################################
#             CONVERT TO TABLE AND KEEP ONLY WANTED VARIABLE
#########################################################################

dataheaders = [i[1:-1] for i in sample_and_hold.columns]
dataheaders.extend(['Shock_Index', 'PaO2_FiO2'])
dataheaders = ['bloc', 'icustayid', 'charttime', 'gender', 'age', 'elixhauser', 're_admission', 'presumed_onset', 'died_in_hosp', \
    'died_within_48h_of_out_time', 'mortality_90d', 'delay_end_of_record_and_discharge_or_death'] + \
    dataheaders + ['median_dose_vaso','max_dose_vaso','input_total','input_4hourly','output_total','output_4hourly','cumulated_balance', 'clinical_note']


reformat2t = pd.DataFrame(reformat2, columns=dataheaders)

In [ ]:
len(dataheaders)

In [ ]:
dataheaders

In [ ]:
# headers I want to keep
dataheaders5 = ['bloc', 'icustayid', 'charttime', 'gender', 'age', 'elixhauser', 're_admission', 'presumed_onset', 'died_in_hosp', 'died_within_48h_of_out_time', \
    'mortality_90d', 'delay_end_of_record_and_discharge_or_death', 'Weight_kg', 'GCS', 'HR', 'SysBP', 'MeanBP', 'DiaBP', 'RR', 'SpO2', 'Temp_C', 'FiO2_1', \
    'Potassium', 'Sodium', 'Chloride', 'Glucose', 'BUN', 'Creatinine', 'Magnesium', 'Calcium', 'Ionised_Ca', 'CO2_mEqL', 'SGOT', 'SGPT', 'Total_bili', 'Albumin', \
    'Hb', 'WBC_count', 'Platelets_count', 'PTT', 'PT', 'INR', 'Arterial_pH', 'paO2', 'paCO2', 'Arterial_BE', 'HCO3', 'Arterial_lactate', 'mechvent', 'Shock_Index', \
    'PaO2_FiO2', 'median_dose_vaso', 'max_dose_vaso', 'input_total', 'input_4hourly', 'output_total', 'output_4hourly', 'cumulated_balance', 'clinical_note']

In [ ]:
reformat3t = reformat2t[dataheaders5].copy()

In [ ]:
## SOME DATA MANIPULATION BEFORE IMPUTATION
#CORRECT GENDER
reformat3t['gender'] = reformat3t['gender'] - 1

# CORRECT AGE > 200 yo
ii = reformat3t['age'] > 150*365.25
reformat3t.loc[ii,'age'] = 91.4*365.25

# FIX MECHVENT
reformat3t['mechvent'].fillna(0, inplace=True)
reformat3t.loc[reformat3t['mechvent'] > 0, 'mechvent'] = 1

# FIX Elixhauser missing values
reformat3t['elixhauser'].loc[np.isnan(reformat3t['elixhauser'].astype(float))] = np.nanmedian(reformat3t['elixhauser'])   #use the median value / only a few missing data points 

# Vasopressors / no NAN
reformat3t['median_dose_vaso'].fillna(0, inplace=True)
reformat3t['max_dose_vaso'].fillna(0, inplace=True)

# clinical note / no NAN
reformat3t['clinical_note'].fillna(0, inplace=True)

# Check missingness proportions here
# miss = pd.DataFrame([np.sum(np.isnan(reformat3t.values), axis=0)/reformat3t.shape[0]], columns=reformat3t.columns)
# Fill the values temporarily with zeros, otherwise kNN imp doesnt work
reformat3t['Shock_Index'] = np.zeros(reformat3t.shape[0])  ## The filling order differs from the earlier block: apply SAH to sysBP and diaBP before calculating shock_index.
reformat3t['PaO2_FiO2'] = np.zeros(reformat3t.shape[0])

In [ ]:
reformat3t

In [ ]:
########################################################################
#        HANDLING OF MISSING VALUES & CREATE REFORMAT4T
########################################################################

# Do linear interpolation where missingness is low (kNN imputation doesnt work if all rows have missing values)
reformat3 = reformat3t.values

In [ ]:
reformat3.shape

In [ ]:
reformat3[:,:58]

In [ ]:
miss = np.sum(np.isnan(reformat3[:,:58].astype(float)), axis=0)/reformat3.shape[0]

In [ ]:
miss

In [ ]:
ii = (miss>0) & (miss<0.05)  #less than 5% missingness
mechventcol = reformat3t.columns.tolist().index('mechvent')

In [ ]:
for i in range(12, mechventcol): # correct col by col, otherwise it does it wrongly
    if ii[i]==1:
        reformat3[:, i] = fixgaps(reformat3[:, i].astype(float))

In [ ]:
reformat3t[reformat3t.columns[12:mechventcol]] = reformat3[:, 12:mechventcol]

In [ ]:
# KNN IMPUTATION - Done on chunks of 10K records.
print('Sepsis Cohort -- KNN imputation with K = 1')

reformat3t_cols = reformat3t.columns.tolist()
mechventcol = reformat3t_cols.index('mechvent')
ref = np.copy(reformat3[:,12:mechventcol])  #columns of interest

bar_knn = pyprind.ProgBar(len(range(0,reformat3.shape[0],9999)))
for i in range(0,reformat3.shape[0],9999):   # Dataset divided in 10K rows chunks (otherwise too large)
    bar_knn.update()
    ref[i:i+9999,:] = KNN(k=1).fit_transform(ref[i:i+9999,:])

# Copy on the interpolated data
reformat3t[reformat3t_cols[12:mechventcol]] = ref 

In [ ]:
reformat4t = reformat3t.copy()  # Make final table copy to compute derived features

In [ ]:
reformat4t

In [ ]:
########################################################################
#        COMPUTE SOME DERIVED FEATURES: P/F, Shock Index, SOFA, SIRS...
########################################################################
# Recompute P/F with no missing values...
reformat4t['PaO2_FiO2'] = reformat4t['paO2']/reformat4t['FiO2_1']

In [ ]:
# Recompute SHOCK INDEX without NAN and INF
reformat4t['Shock_Index'] = reformat4t['HR'].astype(float)/reformat4t['SysBP'].astype(float)

In [ ]:
reformat4t.loc[np.isinf(reformat4t['Shock_Index']), 'Shock_Index'] = np.nan
d = np.nanmean(reformat4t['Shock_Index'])
reformat4t['Shock_Index'].fillna(d, inplace=True)

In [ ]:
shock_idx = reformat4t['Shock_Index'] >= np.quantile(reformat4t['Shock_Index'], 0.999)
reformat4t.loc[shock_idx, 'Shock_Index'] = np.quantile(reformat4t['Shock_Index'], 0.999)

# SOFA - at each timepoint we need (in this order):  
# P/F,  MV,  PLT,  TOT_BILI,  MAP,  NORAD(max),  GCS, CR,  UO
s = reformat4t[['PaO2_FiO2', 'Platelets_count', 'Total_bili', 'MeanBP', 'max_dose_vaso', 'GCS', 'Creatinine', 'output_4hourly']].values
p = np.arange(5)
s1=np.array([s[:, 0]>400, (s[:, 0]>=300) & (s[:, 0]<400), (s[:, 0]>=200) & (s[:, 0]<300), (s[:, 0]>=100) & (s[:, 0]<200), s[:, 0]<100 ])   #count of points for all 6 criteria of sofa
s2=np.array([s[:, 1]>150, (s[:, 1]>=100) & (s[:, 1]<150), (s[:, 1]>=50) & (s[:, 1]<100), (s[:, 1]>=20) & (s[:, 1]<50), s[:, 1]<20 ])
s3=np.array([s[:, 2]<1.2, (s[:, 2]>=1.2) & (s[:, 2]<2), (s[:, 2]>=2) & (s[:, 2]<6), (s[:, 2]>=6) & (s[:, 2]<12), s[:, 2]>12 ])
s4=np.array([s[:, 3]>=70, (s[:, 3]<70) & (s[:, 3]>=65), (s[:, 3]<65), (s[:, 4]>0) & (s[:, 4]<=0.1), s[:, 4]>0.1 ])
s5=np.array([s[:,5]>14, (s[:, 5]>12) & (s[:, 5]<=14), (s[:, 5]>9) & (s[:, 5]<=12), (s[:, 5]>5) & (s[:, 5]<=9), s[:, 5]<=5])
s6=np.array([s[:,6]<1.2, (s[:,6 ]>=1.2) & (s[:, 6]<2), (s[:, 6]>=2) & (s[:, 6]<3.5), ((s[:, 6]>=3.5) & (s[:, 6]<5))|(s[:, 7]<84), (s[:, 6]>5)|(s[:, 7]<34)])

num_columns = reformat4t.shape[1]   # Number of variables in data
newcols_reformat4 = np.zeros((reformat4t.shape[0],7))
for i in range(reformat4t.shape[0]): 
    t = max(p[s1[:,i]], default=0) + max(p[s2[:,i]], default=0) + max(p[s3[:,i]], default=0) + max(p[s4[:,i]], default=0) + max(p[s5[:,i]], default=0) + max(p[s6[:,i]], default=0)  #SUM OF ALL 6 CRITERIA
    if t > 0:
        newcols_reformat4[i,:] = [max(p[s1[:,i]], default=0), max(p[s2[:,i]], default=0), max(p[s3[:,i]], default=0), max(p[s4[:,i]], default=0), max(p[s5[:,i]], default=0), max(p[s6[:,i]], default=0), t]

# SIRS - at each timepoint |  need: temp HR RR PaCO2 WBC 
s = reformat4t[['Temp_C', 'HR', 'RR', 'paCO2', 'WBC_count']].values

s1=np.array([(s[:,0]>=38) | (s[:,0]<=36)])   #count of points for all criteria of SIRS
s2=np.array([s[:,1]>90])
s3=np.array([(s[:,2]>=20) | (s[:,3]<=32)])
s4=np.array([(s[:,4]>=12) | (s[:,4]<4)])
newcols_sirs = (1*s1) + (1*s2) + (1*s3) + (1*s4)

# more IO corrections
reformat4t.loc[reformat4t['input_total'] < 0, 'input_total'] = 0
reformat4t.loc[reformat4t['input_4hourly'] < 0, 'input_4hourly'] = 0

# records values
reformat4t['SOFA'] = newcols_reformat4[:,-1]
reformat4t['SIRS'] = newcols_sirs[0]


In [ ]:
MIMICtable = reformat4t.copy()
MIMICtable.to_csv('./preprocessed_files_with_note/preMIMICtable.csv', index=False) # This matches the original table.

# 19. 분석

In [ ]:
MIMICtable = pd.read_csv('./preprocessed_files_with_note/preMIMICtable.csv')
MIMICtable['age'] = MIMICtable['age']/365

- IV fluid 이상치 제거 2000이상 기록이 있는 icustayid

In [ ]:
print(len(MIMICtable['icustayid'].unique()))
MIMICtable.loc[MIMICtable['clinical_note']=='0', 'clinical_note']='None'

In [ ]:
iv_outlier_id = MIMICtable.loc[MIMICtable['input_4hourly']>=2000, 'icustayid'].unique()
MIMICtable = MIMICtable.loc[~MIMICtable['icustayid'].isin(iv_outlier_id)]
print("filtered icustayid:",len(MIMICtable['icustayid'].unique()))

- episode가 최소 7이상인 경우만 남김

In [ ]:
MIMICtable = MIMICtable.groupby(['icustayid']).filter(lambda x: len(x)>=7)
print("filtered icustayid:",len(MIMICtable['icustayid'].unique()))

- vaso 이상치 icustayid 제거

In [ ]:
vaso_outlier_id = MIMICtable.loc[MIMICtable['max_dose_vaso']>3, 'icustayid'].unique()
MIMICtable = MIMICtable.loc[~MIMICtable['icustayid'].isin(vaso_outlier_id)]
print("filtered icustayid:",len(MIMICtable['icustayid'].unique()))

- clinical note가 episode당 최소 3개 이상 있는것만 남김

In [ ]:
df_note = MIMICtable[['bloc', 'icustayid', 'clinical_note']]
test = df_note.loc[df_note['clinical_note']!='None']
text_counts_per_id = test.groupby('icustayid')['clinical_note'].count()
ids_to_keep = text_counts_per_id[text_counts_per_id >= 3].index
MIMICtable = MIMICtable.loc[MIMICtable['icustayid'].isin(ids_to_keep)]
print("filtered icustayid:",len(MIMICtable['icustayid'].unique()))

- 분석 시작

In [ ]:
print(MIMICtable.loc[MIMICtable['input_4hourly']!=0, 'input_4hourly'].mean())
print(MIMICtable.loc[MIMICtable['input_4hourly']!=0, 'input_4hourly'].std())
print(MIMICtable.loc[MIMICtable['input_4hourly']!=0, 'input_4hourly'].median())

In [ ]:
print(MIMICtable.loc[MIMICtable['max_dose_vaso']!=0, 'max_dose_vaso'].mean())
print(MIMICtable.loc[MIMICtable['max_dose_vaso']!=0, 'max_dose_vaso'].std())
print(MIMICtable.loc[MIMICtable['max_dose_vaso']!=0, 'max_dose_vaso'].median())

In [ ]:
print(MIMICtable['mortality_90d'].mean())
print(MIMICtable['mortality_90d'].std())
print(MIMICtable['mortality_90d'].median())

In [ ]:
for i in list(MIMICtable.columns):
    if i == 'clinical_note':
        pass
    else:
        print(i,"-mean :",MIMICtable[i].mean())
        print(i,"-std :",MIMICtable[i].std())
        print(i,"-median :",MIMICtable[i].median())
        print("---")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df_to_plot = pd.DataFrame(MIMICtable.loc[MIMICtable['input_4hourly'] != 0, "input_4hourly"])
df_to_plot['input_4hourly'] = pd.to_numeric(df_to_plot['input_4hourly'],errors='coerce')
print(df_to_plot['input_4hourly'].mean())
print(df_to_plot['input_4hourly'].std())
print(df_to_plot['input_4hourly'].median())
plt.figure(figsize=(3, 4)) # Optionally adjust the figure size.
ax = df_to_plot.boxplot(showfliers=False) # Store the axes object in ax.
ax.yaxis.set_major_locator(mticker.MultipleLocator(100))
ax.yaxis.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.set_title('IV fluid (4h)')
ax.set_ylabel('ml')
plt.show()

In [ ]:
df_to_plot = pd.DataFrame(MIMICtable.loc[MIMICtable['max_dose_vaso'] != 0, "max_dose_vaso"])
print(df_to_plot['max_dose_vaso'].mean())
print(df_to_plot['max_dose_vaso'].std())
print(df_to_plot['max_dose_vaso'].median())
plt.figure(figsize=(3, 4)) # Optionally adjust the figure size.
ax = df_to_plot.boxplot(showfliers=False) # Store the axes object in ax.
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.1))
ax.yaxis.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.set_title('Vasopressor (4h)')
ax.set_ylabel('mcg/kg/min')
plt.show()

In [ ]:
print("IV의 0이 아닌 데이터 비율: ",(len(MIMICtable[MIMICtable['input_4hourly'].values != 0]) / len(MIMICtable['input_4hourly'].values)))
print("vasopressor의 0이 아닌 데이터 비율: ",(len(MIMICtable[MIMICtable['max_dose_vaso'].values != 0]) / len(MIMICtable['max_dose_vaso'].values)))

In [ ]:
target_id = MIMICtable['icustayid'].unique()
rng = np.random.default_rng(seed=42)
sampled_ids = rng.choice(target_id, size=700, replace=False)
sample_table = MIMICtable.loc[MIMICtable['icustayid'].isin(sampled_ids)]

In [ ]:
MIMICtable.columns

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# 1) Convert timestamps and compute relative times.
sample_table['charttime_dt'] = pd.to_datetime(sample_table['charttime'], unit='s')
if 'presumed_onset_dt' not in sample_table.columns:
    onset_times = (
        sample_table[['icustayid', 'presumed_onset']]
        .drop_duplicates('icustayid')
    )
    onset_times['presumed_onset_dt'] = pd.to_datetime(onset_times['presumed_onset'], unit='s')
    sample_table = sample_table.merge(
        onset_times[['icustayid', 'presumed_onset_dt']],
        on='icustayid', how='left'
    )
sample_table['relative_time_hours'] = (
    sample_table['charttime_dt'] - sample_table['presumed_onset_dt']
).dt.total_seconds() / 3600.0

# 2) Prepare the plot.
fig, ax = plt.subplots(figsize=(12, 6))

# 3) Filter the data, excluding zeros and missing values.
df = sample_table.dropna(subset=['relative_time_hours', 'max_dose_vaso'])
df = df[df['max_dose_vaso'] > 0]
x = df['relative_time_hours'].to_numpy()
y = df['max_dose_vaso'].to_numpy()
total_non_zero = len(x)

if total_non_zero > 0:
    # 4) Configure the bins.
    x_min, x_max = np.floor(x.min()), np.ceil(x.max())
    x_bins = np.arange(x_min, x_max + 1, 1)      # One-hour intervals.
    y_bins = np.arange(0, 0.5 + 0.025, 0.025)    # Bins from 0 to 0.5 with a step of 0.025.

    # 5) Compute the two-dimensional histogram and the actual proportions.
    counts, xedges, yedges = np.histogram2d(x, y, bins=[x_bins, y_bins])
    density = counts / total_non_zero          # Divide by the total sample count instead of using counts/40.

    # 6) Fix the color scale from 0 to the largest proportion.
    norm = Normalize(vmin=0, vmax=density.max())

    # 7) Draw the plot using pcolormesh.
    im = ax.pcolormesh(
        xedges, yedges, density.T,
        cmap=cm.Blues,
        shading='auto',
        norm=norm
    )
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=17)  # cbar tick fontsize
    cbar.set_label('Proportion', fontsize=22, labelpad=10)
else:
    ax.text(0.5, 0.5, 'No data to display',
            ha='center', va='center', transform=ax.transAxes)

# 8) Add reference lines, labels, and a title.
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
ax.axvline(0, color='r', linestyle='--',
           label='Presumed Onset')
ax.set_xlabel("Relative Time from Presumed Onset (Hours)", fontsize=22, labelpad=10)
ax.set_ylabel("Max Dose Vaso (mcg/kg/min)", fontsize=22, labelpad=10)
# ax.set_title("Density Heatmap (Proportion): Max Dose Vaso Over Relative Time")
ax.legend(fontsize=20)
ax.grid(False)

# 9) Configure axis limits and ticks.
ax.set_ylim(0, 0.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(4))
ax.set_xlim(max(-24, x_min - 1), min(48, x_max + 1))

plt.tight_layout()
plt.savefig("./plot/mimic3_vaso.png", dpi=600)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
import matplotlib.cm as cm  # Use a colormap.
import math  # Import ceil for rounding up.

# Assume that sample_table or SNUHtable has already been loaded.
# Required columns: 'icustayid', 'timestep', 'onset_time', and 'input_step'.

# 1) Convert timestamps and compute relative times.
sample_table['charttime_dt'] = pd.to_datetime(sample_table['charttime'], unit='s')
if 'presumed_onset_dt' not in sample_table.columns:
    onset_times = (
        sample_table[['icustayid', 'presumed_onset']]
        .drop_duplicates('icustayid')
    )
    onset_times['presumed_onset_dt'] = pd.to_datetime(onset_times['presumed_onset'], unit='s')
    sample_table = sample_table.merge(
        onset_times[['icustayid', 'presumed_onset_dt']],
        on='icustayid', how='left'
    )
sample_table.dropna(subset=['charttime_dt','presumed_onset_dt'], inplace=True)
sample_table['relative_time_hours'] = (
    sample_table['charttime_dt'] - sample_table['presumed_onset_dt']
).dt.total_seconds() / 3600.0

# 2) Prepare the plot.
fig, ax = plt.subplots(figsize=(12, 6))

# 3) Filter the data to input_step > 0.
df = sample_table.dropna(subset=['relative_time_hours','input_4hourly'])
df = df[df['input_4hourly'] > 0]
x = df['relative_time_hours'].to_numpy()
y = df['input_4hourly'].to_numpy()
total_non_zero = len(x)

if total_non_zero > 0:
    # 4) Configure the x-axis bins at one-hour intervals.
    x_min, x_max = np.floor(x.min()), np.ceil(x.max())
    x_bins = np.arange(x_min, x_max + 1, 1)

    # 5) Configure the y-axis bins using the 99th percentile.
    # 5) Configure the y-axis bins using the 99th percentile.
    y_min = 0
    q99 = np.quantile(y, 0.99)            # Only this part has changed.
    y_max = math.ceil(q99) if q99 > 0 else math.ceil(y.max())
    
    # Choose the bin step.
    step = 25 if y_max <= 500 else (50 if y_max <= 1000 else 100)
    y_bins = np.arange(y_min, y_max + step, step)

    # 6) Compute the two-dimensional histogram and convert counts to proportions.
    counts, xedges, yedges = np.histogram2d(x, y, bins=[x_bins, y_bins])
    density = counts / total_non_zero

    # 7) Fix the color scale from 0 to the global maximum.
    norm = Normalize(vmin=0, vmax=density.max())

    # 8) Draw the plot using pcolormesh.
    im = ax.pcolormesh(
        xedges, yedges, density.T,
        cmap=cm.Reds,
        shading='auto',
        norm=norm
    )
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=17)  # cbar tick fontsize
    cbar.set_label('Proportion', fontsize=22, labelpad=10)
else:
    ax.text(
        0.5, 0.5, 'No data to display',
        ha='center', va='center',
        transform=ax.transAxes
    )

# 9) Draw the presumed-onset reference line.
# 10) Configure labels and the title.
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
ax.axvline(0, color='r', linestyle='--',
           label='Presumed Onset')
ax.set_xlabel("Relative Time from Presumed Onset (Hours)", fontsize=22, labelpad=10)
ax.set_ylabel("IV Fluid Input (ml)", fontsize=22, labelpad=10)
ax.legend(fontsize=20)
ax.grid(False)

# 11) Configure axis limits and ticks.
ax.set_xlim(max(-24, x_min - 1), min(48, x_max + 1))
ax.set_ylim(y_min, y_max)
ax.xaxis.set_major_locator(ticker.MultipleLocator(4))

plt.tight_layout()
plt.savefig("./plot/mimic3_iv.png", dpi=600)
plt.show()

# 19.5 analyze clinical note

In [ ]:
df_note = MIMICtable[['bloc', 'icustayid', 'clinical_note']]
df_note['category'] = df_note['clinical_note'].str.extract(r'\[(.*?)\]', expand=False)
df_note['category'] = df_note['category'].fillna('None')

In [ ]:
! pip install seaborn

In [ ]:
import seaborn as sns
# Exclude the 'Unknown' category.
# Create a new DataFrame containing only rows whose 'category' is not 'Unknown'.
df_filtered = df_note[df_note['category'] != 'None'].copy() # Use .copy() to create an independent DataFrame.

# Recalculate category ordering from the filtered data.
# Use value_counts() on the filtered data to determine the bar order.
filtered_counts = df_filtered['category'].value_counts()
print("--- 'Unknown' 제외 후 카테고리별 텍스트 수 ---")
print(filtered_counts)
print("-" * 30)


In [ ]:
import seaborn as sns
# Exclude the 'Unknown' category.
# Create a new DataFrame containing only rows whose 'category' is not 'Unknown'.
df_filtered = df_note[df_note['category'] != 'None'].copy() # Use .copy() to create an independent DataFrame.

# Recalculate category ordering from the filtered data.
# Use value_counts() on the filtered data to determine the bar order.
filtered_counts = df_filtered['category'].value_counts()
print("--- 'Unknown' 제외 후 카테고리별 텍스트 수 ---")
print(filtered_counts)
print("-" * 30)


# Draw the plot using the updated data and order.
plt.figure(figsize=(12, 8))

# data=df_filtered uses the data after excluding 'Unknown'.
# order=filtered_counts.index uses the ordering calculated from the filtered data.
ax_v = sns.countplot(data=df_filtered,
                     x='category',
                     order=filtered_counts.index, # Apply the ordering of the filtered data.
                     palette='pastel')

# Increase the label and font sizes.
ax_v.set_xlabel('Category', fontsize=28, labelpad=10)
ax_v.set_ylabel('Number of clinical notes', fontsize=28, labelpad=10)

# Set the x-axis tick font size and rotation.
ax_v.tick_params(axis='x', labelsize=19, rotation=45)
ax_v.tick_params(axis='y', labelsize=19)

# Display counts above the bars using the existing behavior.
for patch in ax_v.patches:
    height = patch.get_height()
    x = patch.get_x() + patch.get_width() / 2
    ax_v.text(x,
              height + 0.1, # Leave a small gap above each bar for its count label.
              f'{int(height)}',
              ha='center',
              va='bottom', fontsize=15)

plt.tight_layout() # Adjust the layout.
plt.savefig("./plot/notes_count.png", dpi=600)
plt.show()

In [ ]:
# 3) Count observations in each category.
category_counts = df_note['category'].value_counts()

# -----------------------------------------------------
# 3.5) Group categories representing at most 2.6% into 'Etc'.
# -----------------------------------------------------
threshold_percent = 2.6  # Set the percentage threshold.
total_count = category_counts.sum() # Compute the total count.

# Convert the percentage threshold to a count; group categories at or below this count.
# Allow for floating-point rounding; very small differences are usually negligible.
threshold_count = total_count * (threshold_percent / 100.0)

# Retain categories whose counts are strictly greater than threshold_count.
main_categories = category_counts[category_counts > threshold_count]

# Find categories whose counts are at or below the threshold.
small_categories = category_counts[category_counts <= threshold_count]

# Check whether any categories need to be grouped.
if not small_categories.empty:
    # Sum the small-category counts into 'Etc'.
    etc_count = small_categories.sum()
    # Create a Pandas Series for the 'Etc' category.
    etc_series = pd.Series([etc_count], index=['Etc'])
    # Combine the main categories with 'Etc'.
    # Use pd.concat with recent Pandas versions because Series.append is deprecated.
    plot_counts = pd.concat([main_categories, etc_series])
    # Optionally sort counts in descending order to inspect the results.
    plot_counts = plot_counts.sort_values(ascending=False)
else:
    # Use the original counts when there are no small categories.
    plot_counts = category_counts.sort_values(ascending=False) # Apply this step to keep the ordering consistent.



# -----------------------------------------------------
# 4) Draw the pie chart with 'Etc' and labels inside the chart.
# -----------------------------------------------------
plt.figure(figsize=(9, 9)) # Set the figure size.

# Use the Set3 colormap with the number of categories after grouping.
colors = cm.Pastel2(np.linspace(0, 1, len(plot_counts)))

# Use the existing function to display both percentages and counts.
def make_autopct(values):
    def my_autopct(pct):
        total = sum(values)
        val = int(round(pct*total/100.0))
        # For very small percentages, such as below 0.1%, display only the count or adjust the label.
        if pct < 0.1:
             return f'({val})' # Display only counts for very small values.
        return f'{pct:.1f}%' # For example: 15.5% (150).
    return my_autopct

# Use plt.pie() with plot_counts.index as the labels.
plt.pie(plot_counts.values,             # Size of each slice, represented by its count.
        labels=plot_counts.index,       # Labels for each slice, including 'Etc'.
        autopct=make_autopct(plot_counts.values), # Apply the function that displays percentages and counts.
        startangle=140,                 # Starting angle.
        colors=colors,                  # Apply the selected colors.
        pctdistance=0.8,               # Position the percentages; 0.6 is the default, and values closer to 1 move them outward.
        labeldistance=1.1,             # Position the labels; 1.1 is the default, and values above 1 move them outward.
        textprops={'fontsize': 19})      # Optionally adjust text properties, such as the font size.

# plt.title(f'Text Distribution by Category')
plt.axis('equal') # Use an equal aspect ratio so the pie chart is circular.
plt.tight_layout() # Adjust the layout.
plt.savefig("./plot/note_pi_distribution.png", dpi=600)
plt.show()

In [ ]:
test = df_note.loc[df_note['clinical_note']!='None']
counts_per_id  = test.groupby(['icustayid'])['clinical_note'].count()
counts_distribution = counts_per_id.value_counts().sort_index()

labels = list(dict(counts_distribution).keys())
counts = list(dict(counts_distribution).values())
x = np.arange(len(labels))

# Compute the median and third quartile.
median_val = counts_per_id.median()         # Median.
q3_val     = counts_per_id.quantile(0.75)   # 75th percentile, or third quartile (Q3).

# Find the x-coordinates corresponding to the median and Q3.
# Interpolate their positions if they do not occur exactly among the labels.
try:
    med_x = labels.index(median_val)
except ValueError:
    med_x = np.interp(median_val, labels, x)
try:
    q3_x = labels.index(q3_val)
except ValueError:
    q3_x = np.interp(q3_val, labels, x)

# Configure the style and figure.
plt.style.use('default')
fig, ax = plt.subplots(figsize=(9, 6))

# Extract colors from the colormap.
colors = plt.cm.GnBu(np.linspace(0.2, 0.9, len(labels)))

# Draw the bars.
bars = ax.bar(x, counts, color=colors, edgecolor='gray', linewidth=0.7, alpha=0.9)

# Configure ticks and labels.
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=17)
ax.set_yticklabels(ax.get_yticks(), fontsize=17)
ax.set_xlabel('Text length', fontsize=20, labelpad=10)
ax.set_ylabel('Number of clinical notes', fontsize=20, labelpad=10)
# ax.set_title('Count distribution by text', fontsize=16, pad=15)

# Configure the grid and spines.
# ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.3)
# for spine in ['top', 'right']:
#     ax.spines[spine].set_visible(False)

# Display values above the bars.
# for bar in bars:
#     h = bar.get_height()
#     ax.text(
#         bar.get_x() + bar.get_width()/2,
#         h + max(counts)*0.01,
#         f'{h:,}',
#         ha='center', va='bottom',
#         fontsize=14
#     )

# Add vertical reference lines for the median and Q3.
ax.axvline(med_x, color='red', linestyle='--', linewidth=1.5,
           label=f'Median = {median_val:.1f}')
ax.axvline(q3_x, color='orange', linestyle=':', linewidth=1.5,
           label=f'Q3 = {q3_val:.1f}')

# Add the legend.
ax.legend(fontsize=17)
plt.tight_layout()
plt.savefig("./plot/notes_length.png", dpi=600)
plt.show()

In [ ]:
text_df = df_note.loc[df_note['clinical_note']!="None"]
text_df['text_length'] = text_df['clinical_note'].fillna('').str.len()

In [ ]:
# 1) Remove NaN values and extract the sequence-length series.
lengths = text_df['text_length'].dropna()

# 2) Compute the interquartile range (IQR).
Q1 = lengths.quantile(0.25)
Q3 = lengths.quantile(0.75)
IQR = Q3 - Q1

# 3) Define the outlier bounds.
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 4) Remove outliers.
filtered = lengths[(lengths >= lower_bound) & (lengths <= upper_bound)]

# 5) Draw a box plot with a legend.
plt.figure(figsize=(8, 5), dpi=100)
bp = plt.boxplot(
    filtered,
    showfliers=False,
    showmeans=True,
    meanline=True,
    meanprops={'color': 'red'}
)

# Label the median and mean lines.
bp['means'][0].set_label('Mean')
bp['medians'][0].set_label('Median')

# Display the legend.
plt.legend(loc='best')

plt.title('MIMIC3 text distribution')
plt.xlabel('text')
plt.ylabel('length')
plt.grid(axis='x', linestyle='--', linewidth=0.5)
plt.show()

# 20. Normalization

In [ ]:
# noninterest columns = [bloc, elixhauser, died_in_hosp, died_within_28h_of_out_time, 
                       # mortality_90d, delay_end_of_record_and_discharge_or_death, Ionised_Ca, Co2_mEqL, Albumin, median_dose_vaso]abs

# all 47 columns of interest + additional meta columns to easier associate trajectories with other patient auxiliary info (eg. notes)
colmeta = ['charttime', 'icustayid', 'clinical_note']  # Meta-data around patient stay
colbin = ['gender', 'max_dose_vaso', 're_admission']  # Binary features
# Patient features that will be z-normalize
colnorm = ['age', 'Weight_kg', 'GCS', 'HR', 'SysBP', 'MeanBP', 'DiaBP', 'RR', 'Temp_C', 'FiO2_1', 
        'Potassium', 'Sodium', 'Chloride', 'Glucose', 'Magnesium', 'Calcium', 'Ionised_Ca', 'CO2_mEqL', 'Hb', 
        'WBC_count', 'Platelets_count', 'PTT', 'PT', 'Arterial_pH', 'paO2', 'paCO2',
        'HCO3', 'Arterial_lactate', 'SOFA', 'SIRS', 'Shock_Index',
        'PaO2_FiO2', 'SpO2', 'BUN', 'Creatinine', 'SGOT', 'SGPT', 'Total_bili', 'Albumin', 'INR',
        'input_4hourly', 'output_4hourly']

In [ ]:
import joblib
# find patients who died in ICU during data collection period
icustayidlist = MIMICtable['icustayid']
icuuniqueids = icustayidlist.unique() #list of unique icustayids from MIMIC

MIMICraw = MIMICtable[colmeta + colbin + colnorm]
MIMICraw = MIMICraw.values  # RAW values

from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler = StandardScaler()
z_normalized_colnorm = scaler.fit_transform(MIMICtable[colnorm].values)

# Save to 'scaler.pkl'.
joblib.dump(scaler, './scaler/scaler_mimic3.pkl')

# Clip each value to the interval [-3, 3].
z_clipped = np.clip(z_normalized_colnorm, -3, 3)

# Apply min-max normalization to the interval [0, 1].
minmax_scaler = MinMaxScaler(feature_range=(0, 1))
z_minmax = minmax_scaler.fit_transform(z_clipped)


MIMICzs = np.hstack([MIMICtable[colmeta].values, 
                     MIMICtable[colbin].values, 
                     z_minmax
                    ])

MIMICzs[:,5] = np.log(MIMICzs[:, 5].astype(float) + 0.6)  # Scale MAX DOSE NORAD as in the original workflow; this feature is not used here.
# MIMICzs[:,46] = 2*MIMICzs[:,46]  # Increase weight of this variable ??????

In [ ]:
from scipy import stats

In [ ]:
from scipy import stats
from collections import Counter
# compute conversion factors using MIMIC data
# a = MIMICtable['input_4hourly'].values  # IV fluid
# a = stats.rankdata(a[a>0])/len(a[a>0])  # Rank positive fluid values in ascending order; zero fluid remains action 1.
# iof = np.floor((a+0.2499999999)*4).astype(np.int64) #converts iv volume in 4 actions
# mask_pos = MIMICtable['input_4hourly'] > 0 # location of non-zero fluid in big matrix
# io = np.ones(MIMICtable.shape[0]) # array of ones, by default     
# io[mask_pos] = iof + 1 # where more than zero fluid given: save actual action

# Counter(io)


In [ ]:
# Array of original values.
fluid = MIMICtable['input_4hourly'].values
# Default action 1 represents no fluid.
io = np.ones_like(fluid, dtype=np.int64)
io[(fluid > 0) & (fluid <= 50)] = 2
io[(fluid > 50) & (fluid <= 180)] = 3
io[(fluid > 180) & (fluid <= 320)] = 4
io[fluid > 320] = 5

Counter(io)

In [ ]:
# vc = MIMICtable['max_dose_vaso'].values.copy()
# vcr = stats.rankdata(vc[vc != 0])/len(vc[vc != 0])
# vcr = np.floor((vcr+0.249999999999)*4)
# vcr[vcr == 0] = 1
# vc[vc != 0] = vcr + 1
# vc[vc == 0] = 1

# Counter(vc)

In [ ]:
# Array of original values.
vaso = MIMICtable['max_dose_vaso'].values
# Default action 1 represents no fluid.
vc = np.ones_like(vaso, dtype=np.int64)
vc[(vaso > 0) & (vaso <= 0.05)] = 2
vc[(vaso > 0.05) & (vaso <= 0.10)] = 3
vc[(vaso > 0.10) & (vaso <= 0.20)] = 4
vc[vaso > 0.20] = 5

Counter(vc)

In [ ]:
actions = (io-1)*5 + (vc-1)

In [ ]:
import matplotlib.patches as mpatches
unique_values, counts = np.unique(io, return_counts=True)

# Visualize the distribution with a bar chart.
plt.figure(figsize=(8, 6))
plt.bar(unique_values, counts, width=0.8, color="#840711")
plt.xlabel("IV Fluid Discrete Action", fontsize=17, labelpad=10)
plt.ylabel("Frequency", fontsize=17, labelpad=10)
plt.xticks(unique_values, fontsize=14)  # Display each unique value on the x-axis.
plt.yticks(fontsize=14)
plt.grid(alpha=0.2)
# Legend mapping for the categories.
legend_labels = [
    "1: 0 ml",
    "2: 0~50 ml",
    "3: 50~180 ml",
    "4: 180~320 ml",
    "5: >320 ml",
]
# Create dummy legend patches with the matching colors.
handles = [mpatches.Patch(facecolor="#840711", edgecolor="#840711", label=lab)
           for lab in legend_labels]

plt.legend(handles=handles, fontsize=14, frameon=True, loc="best")
plt.tight_layout()
plt.savefig("./plot/discrete_iv.png", dpi=600)
plt.show()

In [ ]:
MIMICtable["discret_IV"] = io
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==1,"input_4hourly"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==1,"input_4hourly"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==1,"input_4hourly"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==2,"input_4hourly"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==2,"input_4hourly"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==2,"input_4hourly"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==3,"input_4hourly"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==3,"input_4hourly"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==3,"input_4hourly"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==4,"input_4hourly"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==4,"input_4hourly"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==4,"input_4hourly"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==5,"input_4hourly"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==5,"input_4hourly"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==5,"input_4hourly"]))
MIMICtable.drop('discret_IV', axis=1, inplace=True)

In [ ]:
vc[(vaso > 0) & (vaso <= 0.05)] = 2
vc[(vaso > 0.05) & (vaso <= 0.10)] = 3
vc[(vaso > 0.10) & (vaso <= 0.20)] = 4
vc[vaso > 0.20] = 5

In [ ]:
unique_values, counts = np.unique(vc, return_counts=True)

# Visualize the distribution with a bar chart.
plt.figure(figsize=(8, 6))
plt.bar(unique_values, counts, width=0.8, color="#08468B")
plt.xlabel("Vasopressor Discrete Action", fontsize=17, labelpad=10)
plt.ylabel("Frequency", fontsize=17, labelpad=10)
# plt.title("Distribution of Values")
plt.xticks(unique_values, fontsize=14)  # Display each unique value on the x-axis.
plt.yticks(fontsize=14)
plt.grid(alpha=0.2)
# Legend mapping for the categories.
legend_labels = [
    "1: 0 mcg/kg/min",
    "2: 0~0.05 mcg/kg/min",
    "3: 0.05~0.10 mcg/kg/min",
    "4: 0.10~0.20 mcg/kg/min",
    "5: >0.20 mcg/kg/min",
]
# Create dummy legend patches with the matching colors.
handles = [mpatches.Patch(facecolor="#08468B", edgecolor="#08468B", label=lab)
           for lab in legend_labels]

plt.legend(handles=handles, fontsize=14, frameon=True, loc="best")
plt.tight_layout()
plt.savefig("./plot/discrete_vaso.png", dpi=600)
plt.show()

In [ ]:
MIMICtable["discret_vaso"] = vc
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==1,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==1,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==1,"max_dose_vaso"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==2,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==2,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==2,"max_dose_vaso"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==3,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==3,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==3,"max_dose_vaso"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==4,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==4,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==4,"max_dose_vaso"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==5,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==5,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==5,"max_dose_vaso"]))
MIMICtable.drop('discret_vaso', axis=1, inplace=True)

In [ ]:
unique_values, counts = np.unique(actions, return_counts=True)

# Visualize the distribution with a bar chart.
plt.figure(figsize=(8, 4))
plt.bar(unique_values, counts, width=0.8)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.title("Distribution of Values")
plt.xticks(unique_values)  # Display each unique value on the x-axis.
plt.show()

In [ ]:
# 2) Create a DataFrame without column names.

columns = colmeta+colbin+colnorm
col_names = columns[6:]

df = pd.DataFrame(MIMICzs[:, 6:].astype(float), columns=col_names)

plt.figure(figsize=(8, 6))
df.boxplot(grid=False, patch_artist=True, showfliers=False)

# 4) Configure the plot.
plt.title("Box Plot with Mean & Std")
plt.xlabel("Columns")
plt.ylabel("Values")
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Process MIMICzs into trajectory data for later use
MIMICzs             = pd.DataFrame(MIMICzs, columns=colmeta+colbin+colnorm)
meta_df             = pd.DataFrame(MIMICzs[colmeta].values, columns = colmeta)
ob_df               = pd.DataFrame(MIMICzs[colbin+colnorm].values, columns=colbin+colnorm)
ac_df               = pd.DataFrame(actions, columns=['action'])
raw_data_df         = MIMICtable.copy()
num_actions         = 25
outcome_key         = 'mortality_90d' #Should be either 'died_within_48h_of_out_time' or ''mortality_90d''
meta_cols           = meta_df.columns.tolist()
ob_cols             = ob_df.columns.tolist()

In [ ]:
import pyprind
raw_data_df['traj'] = (raw_data_df['bloc'] == 1).cumsum().values
meta_df['traj']     = (raw_data_df['bloc'] == 1).cumsum().values
ob_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
ac_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
trajectories        = raw_data_df['traj'].unique()
data = {}
data['meta_cols'] = meta_cols
data['obs_cols'] = ob_cols
data['traj'] = {}
print('Sepsis Cohort -- Making trajectory data')
bar = pyprind.ProgBar(len(trajectories))
for i in trajectories:
    bar.update()
    data['traj'][i] = {}
    data['traj'][i]['meta'] = meta_df[meta_df['traj']==i][meta_cols].values.T
    data['traj'][i]['obs'] = ob_df[ob_df['traj'] == i][ob_cols].values.T
    data['traj'][i]['actions'] = ac_df[ac_df['traj'] == i]['action'].values.astype(np.int32)
    data['traj'][i]['outcome'] = raw_data_df[raw_data_df['traj'] == i][outcome_key].values[0] 
    data['traj'][i]['rewards'] = np.zeros(len(data['traj'][i]['actions']))
    data['traj'][i]['rewards'][-1] = (1-2*data['traj'][i]['outcome'])


In [ ]:
print('Sepsis Cohort -- Making final output file')
col_names = ['traj', 'step']
col_names.extend(['m:'+ i for i in data['meta_cols']])
col_names.extend(['o:'+ i for i in data['obs_cols']])
col_names.append('a:action')
col_names.append('r:reward')
all_data = []
bar = pyprind.ProgBar(len(data['traj'].keys()))
for i in data['traj'].keys():
    bar.update()
    for ctr in range(data['traj'][i]['actions'].shape[0]):
        all_data.append([])
        all_data[-1].append(i)
        all_data[-1].append(ctr)
        for m_index in range(data['traj'][i]['meta'].shape[0]):
            all_data[-1].append(data['traj'][i]['meta'][m_index, ctr])
        for o_index in range(data['traj'][i]['obs'].shape[0]):
            all_data[-1].append(data['traj'][i]['obs'][o_index, ctr])
        all_data[-1].append(data['traj'][i]['actions'][ctr])
        all_data[-1].append(data['traj'][i]['rewards'][ctr])
df = pd.DataFrame(all_data, columns=col_names)
print("보상은 episode의 마지막에 생존하면 1을 부여하고 죽으면 -1을 부여")

In [ ]:
df.to_csv('./preprocessed_files_with_note/sepsis_final_data_withTimes.csv', index=False)

In [ ]:
raw_df              = pd.DataFrame(MIMICraw, columns=colmeta+colbin+colnorm)
meta_df             = raw_df[colmeta]
meta_df             = pd.DataFrame(meta_df.values, columns=colmeta)
ob_df               = raw_df[colbin+colnorm]
ob_df               = pd.DataFrame(ob_df.values, columns=colbin+colnorm)
ac_df               = pd.DataFrame(actions, columns=['action'])
raw_data_df         = MIMICtable.copy()
num_actions         = 25
outcome_key         = 'mortality_90d' #Should be either 'died_within_48h_of_out_time' or ''mortality_90d''
meta_cols           = meta_df.columns.tolist()
ob_cols             = ob_df.columns.tolist()
raw_data_df['traj'] = (raw_data_df['bloc'] == 1).cumsum().values
meta_df['traj']     = (raw_data_df['bloc'] == 1).cumsum().values
ob_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
ac_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
trajectories        = raw_data_df['traj'].unique()
data = {}
data['meta_cols'] = meta_cols
data['obs_cols'] = ob_cols
data['traj'] = {}
print('Sepsis Cohort -- Making RAW trajectory data')
bar = pyprind.ProgBar(len(trajectories))
for i in trajectories:
    bar.update()
    data['traj'][i] = {}
    data['traj'][i]['meta'] = meta_df[meta_df['traj']==i][meta_cols].values.T
    data['traj'][i]['obs'] = ob_df[ob_df['traj'] == i][ob_cols].values.T
    data['traj'][i]['actions'] = ac_df[ac_df['traj'] == i]['action'].values.astype(np.int32)
    data['traj'][i]['outcome'] = raw_data_df[raw_data_df['traj'] == i][outcome_key].values[0] 
    data['traj'][i]['rewards'] = np.zeros(len(data['traj'][i]['actions']))
    data['traj'][i]['rewards'][-1] = (1-2*data['traj'][i]['outcome'])

print('Sepsis Cohort -- Making final RAW output file')
col_names = ['traj', 'step']
col_names.extend(['m:'+ i for i in data['meta_cols']])
col_names.extend(['o:'+ i for i in data['obs_cols']])
col_names.append('a:action')
col_names.append('r:reward')
all_data = []
bar = pyprind.ProgBar(len(data['traj'].keys()))
for i in data['traj'].keys():
    bar.update()
    for ctr in range(data['traj'][i]['actions'].shape[0]):
        all_data.append([])
        all_data[-1].append(i)
        all_data[-1].append(ctr)
        for m_index in range(data['traj'][i]['meta'].shape[0]):
            all_data[-1].append(data['traj'][i]['meta'][m_index, ctr])            
        for o_index in range(data['traj'][i]['obs'].shape[0]):
            all_data[-1].append(data['traj'][i]['obs'][o_index, ctr])
        all_data[-1].append(data['traj'][i]['actions'][ctr])
        all_data[-1].append(data['traj'][i]['rewards'][ctr])
df = pd.DataFrame(all_data, columns=col_names)



In [ ]:
df.to_csv('./preprocessed_files_with_note/sepsis_final_data_RAW_withTimes.csv', index=False)

## appendix1 : scaling by MIMIC4 & SNUH SCALER

In [ ]:
import joblib
# find patients who died in ICU during data collection period
icustayidlist = MIMICtable['icustayid']
icuuniqueids = icustayidlist.unique() #list of unique icustayids from MIMIC

MIMICraw = MIMICtable[colmeta + colbin + colnorm]
MIMICraw = MIMICraw.values  # RAW values

scaler = joblib.load('./scaler/scaler_snuh.pkl') # mimic3 / snuh
z_normalized_colnorm = scaler.transform(MIMICtable[colnorm].values)


# Clip each value to the interval [-3, 3].
z_clipped = np.clip(z_normalized_colnorm, -3, 3)

# Apply min-max normalization to the interval [0, 1].
minmax_scaler = MinMaxScaler(feature_range=(0, 1))
z_minmax = minmax_scaler.fit_transform(z_clipped)


MIMICzs = np.hstack([MIMICtable[colmeta].values, 
                     MIMICtable[colbin].values, 
                     z_minmax
                    ])

MIMICzs[:,5] = np.log(MIMICzs[:, 5].astype(float) + 0.6)  # Scale MAX DOSE NORAD as in the original workflow; this feature is not used here.
# MIMICzs[:,46] = 2*MIMICzs[:,46]  # Increase weight of this variable ??????

In [ ]:
# Array of original values.
fluid = MIMICtable['input_4hourly'].values
# Default action 1 represents no fluid.
io = np.ones_like(fluid, dtype=np.int64)
# 2 = 0 < fluid ≤ 60
io[(fluid > 0) & (fluid <= 50)] = 2
# 3 = 60 < fluid ≤ 120
io[(fluid > 50) & (fluid <= 180)] = 3
# 4 = 120 < fluid ≤ 320
io[(fluid > 180) & (fluid <= 320)] = 4
# 5 = fluid > 320
io[fluid > 320] = 5

# Array of original values.
vaso = MIMICtable['max_dose_vaso'].values
# Default action 1 represents no fluid.
vc = np.ones_like(vaso, dtype=np.int64)
# 2 = 0 < fluid ≤ 60
vc[(vaso > 0) & (vaso <= 0.05)] = 2
# 3 = 60 < fluid ≤ 120
vc[(vaso > 0.05) & (vaso <= 0.10)] = 3
# 4 = 120 < fluid ≤ 320
vc[(vaso > 0.10) & (vaso <= 0.20)] = 4
# 5 = fluid > 320
vc[vaso > 0.20] = 5

In [ ]:
actions = (io-1)*5 + (vc-1)

In [ ]:
# 2) Create a DataFrame without column names.

columns = colmeta+colbin+colnorm
col_names = columns[6:]

df = pd.DataFrame(MIMICzs[:, 6:].astype(float), columns=col_names)

plt.figure(figsize=(8, 6))
df.boxplot(grid=False, patch_artist=True, showfliers=False)

# 4) Configure the plot.
plt.title("Box Plot with Mean & Std")
plt.xlabel("Columns")
plt.ylabel("Values")
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Process MIMICzs into trajectory data for later use
MIMICzs             = pd.DataFrame(MIMICzs, columns=colmeta+colbin+colnorm)
meta_df             = pd.DataFrame(MIMICzs[colmeta].values, columns = colmeta)
ob_df               = pd.DataFrame(MIMICzs[colbin+colnorm].values, columns=colbin+colnorm)
ac_df               = pd.DataFrame(actions, columns=['action'])
raw_data_df         = MIMICtable.copy()
num_actions         = 25
outcome_key         = 'mortality_90d' #Should be either 'died_within_48h_of_out_time' or ''mortality_90d''
meta_cols           = meta_df.columns.tolist()
ob_cols             = ob_df.columns.tolist()

raw_data_df['traj'] = (raw_data_df['bloc'] == 1).cumsum().values
meta_df['traj']     = (raw_data_df['bloc'] == 1).cumsum().values
ob_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
ac_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
trajectories        = raw_data_df['traj'].unique()
data = {}
data['meta_cols'] = meta_cols
data['obs_cols'] = ob_cols
data['traj'] = {}
print('Sepsis Cohort -- Making trajectory data')
bar = pyprind.ProgBar(len(trajectories))
for i in trajectories:
    bar.update()
    data['traj'][i] = {}
    data['traj'][i]['meta'] = meta_df[meta_df['traj']==i][meta_cols].values.T
    data['traj'][i]['obs'] = ob_df[ob_df['traj'] == i][ob_cols].values.T
    data['traj'][i]['actions'] = ac_df[ac_df['traj'] == i]['action'].values.astype(np.int32)
    data['traj'][i]['outcome'] = raw_data_df[raw_data_df['traj'] == i][outcome_key].values[0] 
    data['traj'][i]['rewards'] = np.zeros(len(data['traj'][i]['actions']))
    data['traj'][i]['rewards'][-1] = (1-2*data['traj'][i]['outcome'])

print('Sepsis Cohort -- Making final output file')
col_names = ['traj', 'step']
col_names.extend(['m:'+ i for i in data['meta_cols']])
col_names.extend(['o:'+ i for i in data['obs_cols']])
col_names.append('a:action')
col_names.append('r:reward')
all_data = []
bar = pyprind.ProgBar(len(data['traj'].keys()))
for i in data['traj'].keys():
    bar.update()
    for ctr in range(data['traj'][i]['actions'].shape[0]):
        all_data.append([])
        all_data[-1].append(i)
        all_data[-1].append(ctr)
        for m_index in range(data['traj'][i]['meta'].shape[0]):
            all_data[-1].append(data['traj'][i]['meta'][m_index, ctr])
        for o_index in range(data['traj'][i]['obs'].shape[0]):
            all_data[-1].append(data['traj'][i]['obs'][o_index, ctr])
        all_data[-1].append(data['traj'][i]['actions'][ctr])
        all_data[-1].append(data['traj'][i]['rewards'][ctr])
df = pd.DataFrame(all_data, columns=col_names)
print("보상은 episode의 마지막에 생존하면 1을 부여하고 죽으면 -1을 부여")


In [ ]:
df.to_csv('./preprocessed_files_with_note/sepsis_final_data_withTimes_scaling_by_snuh.csv', index=False)